In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports
# ──────────────────────────────────────────────────────────────────────────────

import pandas as pd
import re

from pathlib import Path

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 2 — DOMAIN / CARRIER MAPS
# CHANGES: Replaced with full maps covering all significant filers and carriers
# ──────────────────────────────────────────────────────────────────────────────

DOMAIN_MAP = {
    # PE-Backed National
    "halomd.com":               ("HaloMD", "PE-Backed National"),
    "teamhealth.com":           ("TeamHealth", "PE-Backed National"),
    "fam-llc.com":              ("F&A Management", "PE-Backed National"),
    "envisionhealth.com":       ("Envision Healthcare", "PE-Backed National"),
    "scp-health.com":           ("SCP Health", "PE-Backed National"),
    "scphealth.com":            ("SCP Health", "PE-Backed National"),
    "agshealth.com":            ("AGS Health", "PE-Backed National"),
    "totalcare.us":             ("TotalCare / Radiology Partners", "PE-Backed National"),
    "usap.com":                 ("US Anesthesia Partners", "PE-Backed National"),
    "usacs.com":                ("US Acute Care Solutions", "PE-Backed National"),
    "apollomd.com":             ("Apollo MD", "PE-Backed National"),
    "vituity.com":              ("Vituity", "PE-Backed National"),
    "radixhealth.io":           ("Radix Health", "PE-Backed National"),
    "soundphysicians.com":      ("Sound Physicians", "PE-Backed National"),
    "ventrahealth.com":         ("Ventra Health", "PE-Backed National"),
    # Billing / Revenue Cycle
    "saparm.com":               ("Singleton Associates", "Billing/Revenue Cycle"),
    "roundtmc.com":             ("Roundtable Medical Consultants", "Billing/Revenue Cycle"),
    "logixhealth.com":          ("LogixHealth", "Billing/Revenue Cycle"),
    "r1rcm.com":                ("R1 RCM", "Billing/Revenue Cycle"),
    "zotecpartners.com":        ("Zotec Partners", "Billing/Revenue Cycle"),
    "gryphonhc.com":            ("Gryphon Healthcare", "Billing/Revenue Cycle"),
    "qmacsmso.com":             ("QMACS MSO", "Billing/Revenue Cycle"),
    "mdcapitaladvisors.com":    ("MD Capital Advisors", "Billing/Revenue Cycle"),
    "nosurprisebill.com":       ("NoSurpriseBill.com", "Billing/Revenue Cycle"),
    "alldatahealth.com":        ("AllData Health", "Billing/Revenue Cycle"),
    "rightmedicalbilling.com":  ("Right Medical Billing", "Billing/Revenue Cycle"),
    "erevenuebilling.com":      ("eRevenue / Exceptional Health Care", "Billing/Revenue Cycle"),
    "altushealthsystem.com":    ("Altus Health System", "Billing/Revenue Cycle"),
    # Law Firms
    "callagylaw.com":           ("Callagy Law", "Law Firm"),
    "callagyrecovery.com":      ("Callagy Law", "Law Firm"),
    "gottliebandgreenspan.com": ("Gottlieb & Greenspan", "Law Firm"),
    "halkovichlaw.com":         ("Halkovich Law", "Law Firm"),
    "afslaw.com":               ("AFS Law", "Law Firm"),
    "glynnlegal.com":           ("Glynn Legal", "Law Firm"),
    # Hospital Systems
    "primehealthcare.com":      ("Prime Healthcare", "Hospital System"),
    "bmhcc.org":                ("Baptist Memorial Health", "Hospital System"),
    "hcahealthcare.com":        ("HCA Healthcare", "Hospital System"),
    "commonspirit.org":         ("CommonSpirit Health", "Hospital System"),
    "adventhealth.com":         ("AdventHealth", "Hospital System"),
    "wellstar.org":             ("WellStar Health", "Hospital System"),
    "tenethealth.com":          ("Tenet Healthcare", "Hospital System"),
    "saintfrancis.com":         ("Saint Francis Health", "Hospital System"),
    # Specialty Groups
    "sonoranrm.com":            ("Sonoran Radiology", "Specialty Group"),
    "mbbrm.com":                ("Mori Bean & Brooks", "Specialty Group"),
    "empireradrm.com":          ("Empire State Radiology", "Specialty Group"),
    "radpmg.com":               ("Houston Radiology Associates", "Specialty Group"),
    "iairm.com":                ("Imaging Associates of Indiana", "Specialty Group"),
    "specialtycare.net":        ("SpecialtyCare", "Specialty Group"),
    "orthomedstaffing.com":     ("OrthoMed Staffing", "Specialty Group"),
    "nmaiom.com":               ("NMA Intraoperative Monitoring", "Specialty Group"),
    "unitedionm.com":           ("United IONM", "Specialty Group"),
    "pediatrix.com":            ("Pediatrix Medical Group", "Specialty Group"),
    "summit-az.com":            ("Summit AZ", "Specialty Group"),
    "anesthesiadynamics.com":   ("Anesthesia Dynamics", "Specialty Group"),
    "usrcm.net":                ("US Revenue Cycle Mgmt", "Specialty Group"),
}

CARRIER_MAP = {
    "uhc.com":              "UnitedHealthcare",
    "unitedhealthcare.com": "UnitedHealthcare",
    "umr.com":              "UnitedHealthcare",
    "clearhs.com":          "ClearHealth Strategies",
    "clearhealthcloud.com": "ClearHealth Strategies",
    "aetna.com":            "Aetna",
    "aetnahealth.com":      "Aetna",
    "anthem.com":           "Anthem",
    "elevance.com":         "Anthem",
    "bcbstx.com":           "BCBS Texas",
    "hcsc.net":             "BCBS Texas",
    "cigna.com":            "Cigna",
    "evernorth.com":        "Cigna",
    "multiplan.com":        "MultiPlan/Claritev",
    "claritev.com":         "MultiPlan/Claritev",
    "bcbsfl.com":           "Florida Blue",
    "floridablue.com":      "Florida Blue",
    "bcbsil.com":           "BCBS Illinois",
    "bcbst.com":            "BCBS Tennessee",
    "bcbsaz.com":           "BCBS Arizona",
    "azblue.com":           "BCBS Arizona",
    "humana.com":           "Humana",
    "zelis.com":            "Zelis",
    "centene.com":          "Centene",
    "highmark.com":         "Highmark",
    "horizonblue.com":      "Horizon BCBS NJ",
    "bcbsnc.com":           "BCBS NC",
    "bcbssc.com":           "BCBS SC",
    "premera.com":          "Premera",
    "geha.com":             "GEHA",
    "ibx.com":              "Independence Blue Cross",
    "lablue.com":           "BCBS Louisiana",
    "bcbsm.com":            "BCBS Michigan",
    "carefirst.com":        "CareFirst BCBS",
    "bcbsok.com":           "BCBS Oklahoma",
    "bcbsks.com":           "BCBS Kansas",
    "kaiserpermanente.org": "Kaiser Permanente",
    "bcbsmn.com":           "BCBS Minnesota",
    "bcbswny.com":          "BCBS Western NY",
    "excellusbcbs.com":     "Excellus BCBS",
    "medmutual.com":        "Medical Mutual",
    "bcbs.com":             "BCBS (other)",
}

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 3 — HELPERS
# CHANGES: _specialty_group — added Surgery, Pathology/Lab; fixed neuro/surg
#          ordering so "neurosurgery" routes to Surgery not Neurology
# ──────────────────────────────────────────────────────────────────────────────

def _clean_domain(raw):

    if pd.isna(raw):
        return None

    raw = str(raw).lower().strip()

    if raw in ("", "nan", "n/a", "n/r", "none"):
        return None

    m = re.match(r"^((?:[a-z0-9\-]+\.)+[a-z]{2,10})", raw)

    return m.group(1) if m else None


def _specialty_group(raw):

    if pd.isna(raw):
        return ""

    v = str(raw).lower()

    if "emergen" in v:                              return "Emergency"
    if "radio" in v or "imaging" in v:             return "Radiology"
    # Check Surgery before Neurology so 'neurosurgery' routes correctly
    if "surg" in v or "orthop" in v:               return "Surgery"
    if "neuro" in v:                               return "Neurology"
    if "anesth" in v:                              return "Anesthesia"
    if "cardio" in v or "cardiac" in v:            return "Cardiology"
    if "path" in v or ("lab" in v and "rehab" not in v): return "Pathology/Lab"
    if "critical" in v or "intensive" in v:        return "Critical Care"
    if "hospital" in v or "facility" in v:         return "Hospital/Facility"

    return "Other"


def _failure_mode(outcome, is_default):

    if str(is_default).strip() == "Yes":
        return "Default"

    o = str(outcome).lower()

    if "provider" in o or "facility" in o:
        return "Merit Loss"

    if "plan" in o or "issuer" in o:
        return "Carrier Win"

    return ""


def _detect_quarter_from_filename(path: Path):

    name = path.stem.lower()

    m = re.search(r"20(\d\d)[\s_\-]?q(\d)", name)

    if m:
        return f"20{m.group(1)}Q{m.group(2)}"

    return None

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4 — COLUMN NORMALIZATION
# CHANGES:
#   OON — added: Health Plan Type -> plan_type
#                Initiating Party -> initiating_party
#                Provider/Facility NPI Number -> provider_npi
#                Practice/Facility Size -> provider_size
#                Item or Service Description -> service_description
#                Type of Service Code -> service_code_type
#                Dispute Line Item Type -> dispute_line_item_type
#                QPA as Percent of Median QPA -> qpa_pct_median_qpa
#                IDRE Compensation -> idre_compensation
#                Health Plan/Issuer Offer as Percent of Median... -> plan_offer_pct_median
#                Provider/Facility Offer as Percent of Median... -> provider_offer_pct_median
#                Prevailing Offer as Percent of Median... -> prevailing_pct_median
#   QPA  — added: Default Decision -> is_default
#                 Initiating Party -> initiating_party
#                 Dispute Line Item Type -> dispute_line_item_type
#                 Air Ambulance Pick-up Location -> air_ambulance_pickup_location
# ──────────────────────────────────────────────────────────────────────────────

_OON_COL_ALIASES = {

    # Identity
    "Dispute Number":                              "dispute_number",
    "DLI Number":                                  "dli_number",
    "Type of Dispute":                             "dispute_type",
    "Dispute Line Item Type":                      "dispute_line_item_type",

    # Outcome
    "Payment Determination Outcome":               "outcome",
    "Default Decision":                            "is_default",
    "Offer Selected from Provider or Issuer":      "offer_selected",
    "Length of Time to Make Determination":        "days_to_determination",
    "IDRE Compensation":                           "idre_compensation",

    # Provider
    "Provider/Facility Group Name":                "provider_group_name",
    "Provider/Facility Name":                      "provider_name",
    "Provider Email Domain":                       "provider_domain",
    "Provider/Facility NPI Number":                "provider_npi",
    "Practice/Facility Size":                      "provider_size",
    "Initiating Party":                            "initiating_party",

    # Carrier / Plan
    "Health Plan/Issuer Name":                     "carrier_name",
    "Health Plan/Issuer Email Domain":             "carrier_domain",
    "Health Plan Type":                            "plan_type",

    # Service
    "Service Code":                                "service_code",
    "Type of Service Code":                        "service_code_type",
    "Place of Service Code":                       "place_of_service",
    "Item or Service Description":                 "service_description",
    "Location of Service":                         "state",
    "Practice/Facility Specialty or Type":         "specialty_raw",

    # Offers as % of QPA — two naming conventions across years
    "Provider/Facility Offer as Percent of QPA":   "offer_pct_qpa_provider",
    "Provider/Facility Offer as % of QPA":         "offer_pct_qpa_provider",
    "Health Plan/Issuer Offer as Percent of QPA":  "offer_pct_qpa_plan",
    "Health Plan/Issuer Offer as % of QPA":        "offer_pct_qpa_plan",
    "Prevailing Party Offer as Percent of QPA":    "offer_pct_qpa_prevailing",
    "Prevailing Party Offer as % of QPA":          "offer_pct_qpa_prevailing",

    # Relative-to-median columns
    "QPA as Percent of Median QPA":                                                    "qpa_pct_median_qpa",
    "Provider/Facility Offer as Percent of Median Provider/Facility Offer Amount":     "provider_offer_pct_median",
    "Health Plan/Issuer Offer as Percent of Median Health Plan/Issuer Offer Amount":   "plan_offer_pct_median",
    "Prevailing Offer as Percent of Median Prevailing Offer Amount":                   "prevailing_pct_median",
}


_QPA_COL_ALIASES = {

    "Service Code":                                "service_code",
    "Type of Service Code":                        "service_code_type",
    "Place of Service Code":                       "place_of_service",
    "Geographical Region":                         "geographic_region",

    "QPA":                                         "qpa",
    "Provider/Facility Offer":                     "provider_offer",
    "Health Plan/Issuer Offer":                    "plan_offer",
    "Prevailing Offer":                            "prevailing_offer",

    "Offer Selected from Provider or Issuer":      "offer_selected",
    "Default Decision":                            "is_default",
    "Initiating Party":                            "initiating_party",
    "Dispute Line Item Type":                      "dispute_line_item_type",
    "Air Ambulance Pick-up Location":              "air_ambulance_pickup_location",
}

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5 — FILE READERS  (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def _read_csv(path):

    print(f"    Reading CSV: {path.name}")

    return pd.read_csv(
        path,
        dtype=str,
        low_memory=False,
        encoding="utf-8",
        encoding_errors="replace"
    )


def _read_xlsx_sheet(path, sheet_name):

    print(f"    Reading sheet '{sheet_name}' from {path.name}")

    return pd.read_excel(
        path,
        sheet_name=sheet_name,
        dtype=str,
        engine="openpyxl"
    )


def _find_oon_sheet(path):

    import openpyxl

    wb = openpyxl.load_workbook(path, read_only=True)

    for s in wb.sheetnames:

        sl = s.lower()

        if "oon" in sl and ("emergency" in sl or "non" in sl):
            wb.close()
            return s

    wb.close()

    return None


def _find_qpa_sheet(path):

    import openpyxl

    wb = openpyxl.load_workbook(path, read_only=True)

    for s in wb.sheetnames:

        if "qpa" in s.lower():
            wb.close()
            return s

    wb.close()

    return None


def _find_air_ambulance_sheet(path):

    import openpyxl

    wb = openpyxl.load_workbook(path, read_only=True)

    for s in wb.sheetnames:

        sl = s.lower()

        if "air" in sl and "ambulance" in sl:
            wb.close()
            return s

    wb.close()

    return None

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 6 — ENRICHMENT
# CHANGES: numeric conversion for offer_pct_qpa_* moved here so it runs per
#          file rather than only at concat time — avoids mixed-type concat
#          warnings on large files. Also strips '%' suffix if present
#          (some quarters store "123%" as a string).
# ──────────────────────────────────────────────────────────────────────────────

def _to_numeric_pct(series):
    """Strip trailing '%' if present, then coerce to float."""
    cleaned = series.astype(str).str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def _enrich_oon(df, quarter, source_file, dataset_type):

    df = df.rename(columns=_OON_COL_ALIASES)

    df.insert(0, "quarter", quarter)

    df["dataset_type"] = dataset_type

    df["source_file"] = source_file

    # Specialty grouping
    if "specialty_raw" in df.columns:
        df["specialty_group"] = df["specialty_raw"].apply(_specialty_group)

    # Failure mode
    df["failure_mode"] = [
        _failure_mode(o, d)
        for o, d in zip(
            df.get("outcome", pd.Series(dtype=str)).fillna(""),
            df.get("is_default", pd.Series(dtype=str)).fillna("")
        )
    ]

    # Provider group + category from domain
    if "provider_domain" in df.columns:

        domains = df["provider_domain"].apply(_clean_domain)

        df["provider_group"] = domains.map(
            lambda d: DOMAIN_MAP.get(d, ("", ""))[0] if d else ""
        )

        df["provider_category"] = domains.map(
            lambda d: DOMAIN_MAP.get(d, ("", ""))[1] if d else ""
        )

    # Carrier group from domain
    if "carrier_domain" in df.columns:

        carrier_domains = df["carrier_domain"].apply(_clean_domain)

        df["carrier_group"] = carrier_domains.map(
            lambda d: CARRIER_MAP.get(d, "") if d else ""
        )

    # Numeric conversion for offer % columns — strip '%' suffix before casting
    for col in ["offer_pct_qpa_provider", "offer_pct_qpa_plan", "offer_pct_qpa_prevailing",
                "qpa_pct_median_qpa", "provider_offer_pct_median",
                "plan_offer_pct_median", "prevailing_pct_median", "days_to_determination"]:
        if col in df.columns:
            df[col] = _to_numeric_pct(df[col])

    return df


def _enrich_qpa(df, quarter, source_file):

    df = df.rename(columns=_QPA_COL_ALIASES)

    df.insert(0, "quarter", quarter)

    df["dataset_type"] = "qpa"

    df["source_file"] = source_file

    # Numeric conversion — '^' (suppressed) and 'N/A' become NaN
    for col in ["qpa", "provider_offer", "plan_offer", "prevailing_offer"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 7 — FILE DISCOVERY  (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def _oon_sources(base):

    sources = []

    for year_dir in sorted(base.glob("[0-9][0-9][0-9][0-9]")):

        year = int(year_dir.name)

        for f in sorted(year_dir.iterdir()):

            name_l = f.name.lower()

            if "supplemental" in name_l:
                continue

            if "qpa" in name_l:
                continue

            if "air ambulance" in name_l:
                continue

            quarter = _detect_quarter_from_filename(f)

            if not quarter:
                continue

            if year < 2025 and f.suffix == ".xlsx":
                sources.append((quarter, f, "xlsx"))

            elif year >= 2025 and f.suffix == ".csv":

                if "oon" in name_l:
                    sources.append((quarter, f, "csv"))

    return sorted(sources)


def _qpa_sources(base):

    sources = []

    for year_dir in sorted(base.glob("[0-9][0-9][0-9][0-9]")):

        year = int(year_dir.name)

        for f in sorted(year_dir.iterdir()):

            name_l = f.name.lower()

            if "supplemental" in name_l:
                continue

            quarter = _detect_quarter_from_filename(f)

            if not quarter:
                continue

            if year < 2025 and f.suffix == ".xlsx":
                sources.append((quarter, f, "xlsx"))

            elif year >= 2025 and f.suffix == ".csv":

                if "qpa" in name_l or "offer" in name_l:
                    sources.append((quarter, f, "csv"))

    return sorted(sources)


def _air_sources(base):

    sources = []

    for year_dir in sorted(base.glob("[0-9][0-9][0-9][0-9]")):

        year = int(year_dir.name)

        for f in sorted(year_dir.iterdir()):

            name_l = f.name.lower()

            if "supplemental" in name_l:
                continue

            quarter = _detect_quarter_from_filename(f)

            if not quarter:
                continue

            if year < 2025 and f.suffix == ".xlsx":
                sources.append((quarter, f, "xlsx"))

            elif year >= 2025 and f.suffix == ".csv":

                if "air ambulance" in name_l:
                    sources.append((quarter, f, "csv"))

    return sorted(sources)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 8 — MAIN LOADERS
# CHANGES: removed duplicate numeric conversion from load_oon/load_qpa since
#          it now happens per-file in _enrich_oon/_enrich_qpa
# ──────────────────────────────────────────────────────────────────────────────

def load_oon(base_path=".", verbose=True):

    base = Path(base_path)

    sources = _oon_sources(base)

    frames = []

    for quarter, path, kind in sources:

        print(f"[{quarter}] {path.name}")

        try:

            if kind == "xlsx":

                sheet = _find_oon_sheet(path)

                if not sheet:
                    print("    No OON sheet found")
                    continue

                df = _read_xlsx_sheet(path, sheet)

            else:

                df = _read_csv(path)

            df = _enrich_oon(df, quarter, path.name, "oon")

            frames.append(df)

            print(f"    → {len(df):,} rows")

        except Exception as e:

            print(f"    ERROR: {e}")

    combined = pd.concat(frames, ignore_index=True, sort=False)

    print()
    print(f"✓ OON master loaded")
    print(f"  Rows:     {len(combined):,}")
    print(f"  Columns:  {len(combined.columns)}")
    print(f"  Quarters: {sorted(combined['quarter'].unique())}")

    return combined


def load_qpa(base_path=".", verbose=True):

    base = Path(base_path)

    sources = _qpa_sources(base)

    if not sources:
        print("WARNING: No QPA files found")
        return pd.DataFrame()

    frames = []

    for quarter, path, kind in sources:

        print(f"[{quarter}] {path.name}")

        try:

            if kind == "xlsx":

                sheet = _find_qpa_sheet(path)

                if not sheet:
                    print("    No QPA sheet found")
                    continue

                df = _read_xlsx_sheet(path, sheet)

            else:

                df = _read_csv(path)

            df = _enrich_qpa(df, quarter, path.name)

            frames.append(df)

            print(f"    → {len(df):,} rows")

        except Exception as e:

            print(f"    ERROR: {e}")

    combined = pd.concat(frames, ignore_index=True, sort=False)

    print()
    print(f"✓ QPA master loaded")
    print(f"  Rows:     {len(combined):,}")
    print(f"  Columns:  {len(combined.columns)}")
    print(f"  Quarters: {sorted(combined['quarter'].unique())}")

    return combined


def load_air_ambulance(base_path=".", verbose=True):

    base = Path(base_path)

    sources = _air_sources(base)

    if not sources:
        print("WARNING: No Air Ambulance files found")
        return pd.DataFrame()

    frames = []

    for quarter, path, kind in sources:

        print(f"[{quarter}] {path.name}")

        try:

            if kind == "xlsx":

                sheet = _find_air_ambulance_sheet(path)

                if not sheet:
                    print("    No Air Ambulance sheet found")
                    continue

                df = _read_xlsx_sheet(path, sheet)

            else:

                df = _read_csv(path)

            df = _enrich_oon(df, quarter, path.name, "air_ambulance")

            frames.append(df)

            print(f"    → {len(df):,} rows")

        except Exception as e:

            print(f"    ERROR: {e}")

    combined = pd.concat(frames, ignore_index=True, sort=False)

    print()
    print(f"✓ Air Ambulance master loaded")
    print(f"  Rows:     {len(combined):,}")
    print(f"  Quarters: {sorted(combined['quarter'].unique())}")

    return combined

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 9 — SUMMARY HELPER
# ──────────────────────────────────────────────────────────────────────────────

def summary(df):

    if "quarter" not in df.columns:

        print("No quarter column found")

        return

    out = (
        df.groupby("quarter")
        .size()
        .reset_index(name="rows")
    )

    print(out.to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 9 — LOAD
# ──────────────────────────────────────────────────────────────────────────────

oon = load_oon(".")

In [ ]:
qpa = load_qpa(".")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 10 — QUICK CHECKS
# ──────────────────────────────────────────────────────────────────────────────

# Confirm all new columns are present
expected = [
    "quarter", "dispute_number", "dli_number", "dispute_type", "dispute_line_item_type",
    "outcome", "is_default", "failure_mode", "days_to_determination", "idre_compensation",
    "provider_name", "provider_group_name", "provider_domain", "provider_npi",
    "provider_size", "initiating_party", "provider_group", "provider_category",
    "carrier_name", "carrier_domain", "carrier_group", "plan_type",
    "service_code", "service_code_type", "place_of_service", "service_description",
    "state", "specialty_raw", "specialty_group",
    "offer_pct_qpa_provider", "offer_pct_qpa_plan", "offer_pct_qpa_prevailing",
]

missing = [c for c in expected if c not in oon.columns]
print("Missing expected columns:", missing if missing else "None — all good ✓")

print("\nplan_type value counts (top 10):")
print(oon["plan_type"].value_counts(dropna=False).head(10))

print("\nfailure_mode value counts:")
print(oon["failure_mode"].value_counts(dropna=False))

print("\ninitiating_party value counts (quarters where available):")
print(oon["initiating_party"].value_counts(dropna=False).head(5))

print("\nRows per quarter:")
print(oon.groupby("quarter").size())

In [ ]:
# "offer_pct_qpa_provider", "offer_pct_qpa_plan", "offer_pct_qpa_prevailing",

non_default = oon[oon['is_default'] != 'Yes']

dispute_level = (
    non_default
    .groupby('dispute_number', as_index=False)['offer_pct_qpa_provider']
    .median()
)

overall_median = dispute_level['offer_pct_qpa_provider'].median()

overall_median

In [ ]:
oon[oon['provider_group'] == 'HaloMD']['provider_domain'].value_counts()

In [ ]:
fees = pd.to_numeric(oon['idre_compensation'], errors='coerce')

print(fees.describe())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 10 — LOAD EVERYTHING
# ──────────────────────────────────────────────────────────────────────────────

print("Loading OON...")
oon = load_oon(".")

print()
print("Loading QPA...")
qpa = load_qpa(".")

print()
print("Loading Air Ambulance...")
air = load_air_ambulance(".")

In [ ]:
oon.to_parquet("oon_processed.parquet")

In [ ]:
!pip install pyarrow

In [ ]:
qpa = load_qpa(".")

In [ ]:
air = load_air_ambulance(".")

In [ ]:
summary(qpa)

In [ ]:
summary(oon)

In [ ]:
summary(air)

In [ ]:
oon[oon['Initiating Party'].notna()].head(5)

In [1]:
import pickle

In [2]:
with open("oon.pkl", "rb") as f:
    oon = pickle.load(f)

In [3]:
with open("qpa.pkl", "rb") as f:
    qpa = pickle.load(f)

In [4]:
with open("air.pkl", "rb") as f:
    air = pickle.load(f)

In [ ]:
oon.groupby('provider_group_name') \
   .agg({'dispute_number': 'nunique'}) \
   .sort_values('dispute_number', ascending=False)

In [ ]:
oon[
    oon['carrier_name']
        .str.contains('anthem|elevance', case=False, na=False)
]['carrier_name'].drop_duplicates()

In [ ]:
anthem = oon[oon['carrier_name'].str.contains('anthem|elevance', case=False, na=False)]

In [ ]:
anthem['plan_type'].unique()

In [ ]:
anthem_self = anthem[anthem['plan_type'].str.contains('self', case=False, na=False)]

In [ ]:
anthem_self['quarter'].unique()

In [ ]:
df = anthem_self[anthem_self['quarter'].isin(['2024Q4', '2025Q1', '2025Q2'])]

In [ ]:
df.columns

In [ ]:
df.to_csv('inspection.csv', index=False)

In [5]:
oon.columns

Index(['quarter', 'dispute_number', 'dli_number', 'outcome', 'is_default',
       'dispute_type', 'provider_group_name', 'provider_name',
       'provider_domain', 'provider_npi', 'provider_size', 'carrier_name',
       'carrier_domain', 'plan_type', 'days_to_determination',
       'idre_compensation', 'dispute_line_item_type', 'service_code_type',
       'service_code', 'place_of_service', 'service_description', 'state',
       'specialty_raw', 'offer_pct_qpa_provider', 'offer_pct_qpa_plan',
       'offer_selected', 'offer_pct_qpa_prevailing', 'qpa_pct_median_qpa',
       'provider_offer_pct_median', 'plan_offer_pct_median',
       'prevailing_pct_median', 'dataset_type', 'source_file',
       'specialty_group', 'failure_mode', 'provider_group',
       'provider_category', 'carrier_group', 'initiating_party'],
      dtype='object')

In [7]:
oon[['provider_group_name', 'provider_name', 'provider_domain', 'provider_group']].head(10)

,provider_group_name,provider_name,provider_domain,provider_group
0,LONGHORN EMERGENCY MEDICAL ASSOCIATES PA,"BLUDWORTH MD,WHITNEY G",teamhealth.com,TeamHealth
1,NR,ROXBOROUGH MEMORIAL HOSPITAL,primehealthcare.com,Prime Healthcare
2,WOODLANDS EMERGENCY PHYSICIANS,WOODLANDS EMERGENCY PHYSICIANS,r1rcm.com,R1 RCM
3,APP OF TENNESSEE ED PLLC,APP OF TENNESSEE ED PLLC,r1rcm.com,R1 RCM
4,Remote Neuromonitoring Physicians PC,Remote Neuromonitoring Physicians PC,specialtycare.net,SpecialtyCare
5,Remote Neuromonitoring Physicians PC,Remote Neuromonitoring Physicians PC,specialtycare.net,SpecialtyCare
6,EMERGIGROUP PHYS ASSOC PLLC,EMERGIGROUP PHYS ASSOC PLLC,r1rcm.com,R1 RCM
7,SAN JACINTO EMER PHYS PLLC,SAN JACINTO EMER PHYS PLLC,r1rcm.com,R1 RCM
8,LONGHORN EMERGENCY MEDICAL ASSOCIATES PA,"KENDALL DO,ANDREW",teamhealth.com,TeamHealth
9,LONGHORN EMERGENCY MEDICAL ASSOCIATES PA,"HOOD MD,STEFAN P",teamhealth.com,TeamHealth


In [10]:
oon[oon['carrier_group'].str.contains('Anthem')]

,quarter,dispute_number,dli_number,outcome,is_default,dispute_type,provider_group_name,provider_name,provider_domain,provider_npi,...,plan_offer_pct_median,prevailing_pct_median,dataset_type,source_file,specialty_group,failure_mode,provider_group,provider_category,carrier_group,initiating_party
101,2023Q1,DISP-135488,DLI - 582330,In Favor of Provider/Facility/AA Provider,No,Single,DELAWARE RIVER MEDICINE,DELAWARE RIVER MEDICINE,r1rcm.com,1457987521,...,0.59,1.00,oon,2023-q1-federal-idr-puf_0.xlsx,Emergency,Merit Loss,R1 RCM,Billing/Revenue Cycle,Anthem,NaN
392,2023Q1,DISP-136631,DLI - 584399,In Favor of Provider/Facility/AA Provider,No,Single,"Kingsford Emergency Group, LLC","Kingsford Emergency Group, LLC",scp-health.com,1336631480,...,0.87,1.01,oon,2023-q1-federal-idr-puf_0.xlsx,Emergency,Merit Loss,SCP Health,PE-Backed National,Anthem,NaN
500,2023Q1,DISP-137278,DLI - 586135,In Favor of Provider/Facility/AA Provider,No,Single,Sky Ridge Medical Center,South Denver Acute Surgery PC,gmail.com,1457590028,...,2.08,5.29,oon,2023-q1-federal-idr-puf_0.xlsx,Surgery,Merit Loss,,,Anthem,NaN
582,2023Q1,DISP-136640,DLI - 584407,In Favor of Provider/Facility/AA Provider,No,Single,"Kingsford Emergency Group, LLC","Kingsford Emergency Group, LLC",scp-health.com,1336631480,...,0.85,1.31,oon,2023-q1-federal-idr-puf_0.xlsx,Emergency,Merit Loss,SCP Health,PE-Backed National,Anthem,NaN
646,2023Q1,DISP-136502,DLI - 584247,In Favor of Provider/Facility/AA Provider,No,Batched,Western Virginia Regional Emergency Physicians...,Western Virginia Regional Emergency Phys,agshealth.com,1386183028,...,0.99,1.00,oon,2023-q1-federal-idr-puf_0.xlsx,Emergency,Merit Loss,AGS Health,PE-Backed National,Anthem,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5253004,2025Q2,DISP-2492394,DLI - 6840796,In Favor of Provider/Facility/AA Provider,Yes,Batched,N/R,Garden Grove Hospital,primehealthcare.com,N/R,...,NaN,1.00,oon,Federal IDR PUF for 2025 Q2 - OON Emergency & ...,Hospital/Facility,Default,Prime Healthcare,Hospital System,Anthem,Health care facility
5253009,2025Q2,DISP-2937366,DLI - 8189706,In Favor of Plan/Issuer,No,Batched,LAKE SPRING EMER GROUP LLC,LAKE SPRING EMER GROUP LLC,agshealth.com,1215429600,...,0.98,0.23,oon,Federal IDR PUF for 2025 Q2 - OON Emergency & ...,Emergency,Carrier Win,AGS Health,PE-Backed National,Anthem,Health care provider
5253039,2025Q2,DISP-2996406,DLI - 8353484,In Favor of Provider/Facility/AA Provider,No,Batched,N/R,St Francis Medical Center,primehealthcare.com,N/R,...,0.00,4.38,oon,Federal IDR PUF for 2025 Q2 - OON Emergency & ...,Hospital/Facility,Merit Loss,Prime Healthcare,Hospital System,Anthem,Health care facility
5253040,2025Q2,DISP-2570265,DLI - 7057687,In Favor of Provider/Facility/AA Provider,No,Single,"ONE CARE MONITORING, LLC","ONE CARE MONITORING, LLC",halomd.com,1083141691,...,0.79,1.00,oon,Federal IDR PUF for 2025 Q2 - OON Emergency & ...,Neurology,Merit Loss,HaloMD,PE-Backed National,Anthem,Health care provider


In [15]:
oon.groupby('carrier_domain').agg({
    'dispute_number': 'nunique'
}).sort_values(by='dispute_number', ascending=False)

,dispute_number
carrier_domain,
uhc.com,649309
aetna.com,357946
bcbstx.com,259490
multiplan.com,245896
anthem.com,194181
...,...
yuzu.health,1
6dgereeshealth.com,1
6degresshealth.com,1


In [17]:
oon.groupby('carrier_domain').agg({
    'dispute_number': 'nunique'
}).sort_values(by='dispute_number', ascending=False).to_csv('temp.csv')

In [20]:
# ==========================================
# ANTHEM / ELEVANCE DOMAIN DETECTION
# ==========================================

anthem_patterns = [
    
    # Core Anthem / Elevance
    r'anthem',
    r'elevance',
    
    # Subsidiaries / brands
    r'amerigroup',
    r'carelon',
    r'caremore',
    r'unicare',
    r'wellpoint',
    r'healthkeepers',
    r'empireblue',
    
    # BCBS entities owned by Anthem
    r'bcbsga',
    
    # Affiliated / TPA
    r'ameriben',
]

# Clean field first
oon['carrier_domain_clean'] = (
    oon['carrier_domain']
    .astype(str)
    .str.lower()
    .str.strip()
)

# Single regex
anthem_regex = '|'.join(anthem_patterns)

# Main flag
oon['anthem'] = np.where(
    oon['carrier_domain_clean'].str.contains(
        anthem_regex,
        regex=True,
        na=False
    ),
    'Anthem',
    'Other'
)

In [22]:
oon[oon['anthem'] == 'Anthem'][['carrier_name', 'carrier_domain', 'carrier_group', 'anthem']].groupby

,carrier_name,carrier_domain,carrier_group,anthem
101,BCBS OF NEW YORK-EMPIRE,anthem.com,Anthem,Anthem
392,Anthem Virginia,anthem.com,Anthem,Anthem
500,Anthem Blue Cross and Blue Shield,anthem.com,Anthem,Anthem
582,Anthem Virginia,anthem.com,Anthem,Anthem
646,Anthem Virginia,anthem.com,Anthem,Anthem
647,Anthem Virginia,anthem.com,Anthem,Anthem
672,BCBS,anthem.com,Anthem,Anthem
724,Anthem Blue Cross Blue Shield,anthem.com,Anthem,Anthem
729,Anthem Blue Cross Blue Shield,anthem.com,Anthem,Anthem
758,Anthem Virginia,anthem.com,Anthem,Anthem


In [24]:
oon[
    oon['anthem'] == 'Anthem'
][[
    # 'carrier_name',
    'carrier_domain',
    'carrier_group',
    'anthem'
]].drop_duplicates()

,carrier_domain,carrier_group,anthem
101,anthem.com,Anthem,Anthem
1928,empireblue.com,,Anthem
36143,ameriben.com,,Anthem
48315,email.anthem.com,,Anthem
157142,anthem.comq,,Anthem
185020,anthem.com800-331-1476,Anthem,Anthem
263387,elevancehealth.com,,Anthem
339189,carelon.com,,Anthem
458843,amerigroup.com,,Anthem
513192,providers.anthem.com,,Anthem


In [26]:
oon[
    oon['anthem'] == 'Anthem'
].groupby([
    # 'carrier_name',
    'carrier_domain',
    'carrier_group',
    'anthem'
]).agg(
    dispute_count=('dispute_number', 'nunique')
).reset_index().sort_values(
    'dispute_count',
    ascending=False
)

,carrier_domain,carrier_group,anthem,dispute_count
3,anthem.com,Anthem,Anthem,194181
15,empireblue.com,,Anthem,1052
0,ameriben.com,,Anthem,931
13,elevancehealth.com,,Anthem,450
19,wellpoint.com,,Anthem,11
12,carelon.com,,Anthem,8
17,unicare.com,,Anthem,6
11,bcbsga.com,,Anthem,5
14,email.anthem.com,,Anthem,5
8,anthem.comq,,Anthem,5


In [27]:
oon.to_parquet('idr_full_2023_2025.parquet', compression='snappy')

In [ ]:
all_dfs = []

for file, quarter, file_type in file_list:
    # 1. Load
    if file_type == 'xlsx':
        df = pd.read_excel(file, sheet_name='OON Emergency and Non-Emergency', engine='openpyxl')
    else:  # csv
        df = pd.read_csv(file)
    
    # 2. Normalize
    df = normalize_oon(df, quarter)
    
    # 3. Append
    all_dfs.append(df)

# 4. Concat once at the end
oon = pd.concat(all_dfs, ignore_index=True)

In [42]:
"""
IDR PUF Data Loader
====================
Loads all quarters (2023 Q1 - 2025 Q2) of CMS Federal IDR Public Use Files
into three consolidated dataframes:
  - oon : OON Emergency & Non-Emergency
  - qpa : QPA and Offers
  - air : OON Air Ambulance

Output files (parquet):
  - oon_all_quarters.parquet
  - qpa_all_quarters.parquet
  - air_all_quarters.parquet

Notes on known data quirks handled here:
  - Column rename: "as Percent of QPA" -> "as % of QPA" (changed in 2023 Q3)
  - New column in 2025 Q1+: "Initiating Party" (will be NaN for 2023/2024 rows)
  - "NR" and "N/R" are CMS suppression/missing markers — replaced with NaN globally
    across all columns (affects Default Decision, NPI Number, and others)
  - 2025 files are CSVs; 2023/2024 are xlsx
  - 2025 Q1 OON split into 2 chunks; 2025 Q2 OON split into 3 chunks
  - TODO: On fresh file download, remove leading space from QPA and Offers sheetname
      in 2023Q2 file (or fix in code to adjust)
      > same file, change ambulance sheet to 'OON Air Ambulance' from 'Air Ambulance'
"""

import pandas as pd
import os

# ── Update this to wherever your files live ───────────────────────────────────
DATA_DIR = r"."   # or e.g. r"C:\Users\you\Downloads\IDR"

OUT_DIR  = r"./parquet"   # where parquets will be saved
os.makedirs(OUT_DIR, exist_ok=True)

def fp(filename):
    """Build full file path."""
    return os.path.join(DATA_DIR, filename)

def out(filename):
    """Build output file path."""
    return os.path.join(OUT_DIR, filename)


# ── File manifest ─────────────────────────────────────────────────────────────
#
# Each entry: (quarter_label, file_type, oon_source, qpa_source, air_source)
#
#   file_type  : 'xlsx' or 'csv'
#   oon_source : filename (xlsx) or list of filenames (csv chunks)
#   qpa_source : filename or None if not available
#   air_source : filename or None if not available

QUARTERS = [
    {
        'quarter'  : '2023Q1',
        'filetype' : 'xlsx',
        'oon'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
        'qpa'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
        'air'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
    },
    {
        'quarter'  : '2023Q2',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
        'qpa'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
        'air'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
    },
    {
        'quarter'  : '2023Q3',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
        'qpa'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
        'air'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
    },
    {
        'quarter'  : '2023Q4',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
        'qpa'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
        'air'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
    },
    {
        'quarter'  : '2024Q1',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
    },
    {
        'quarter'  : '2024Q2',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
    },
    {
        'quarter'  : '2024Q3',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
    },
    {
        'quarter'  : '2024Q4',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
    },
    {
        'quarter'  : '2025Q1',
        'filetype' : 'csv',
        'oon'      : [
            '2025/Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk1.csv',
            '2025/Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk2.csv',
        ],
        'qpa'      : '2025/Federal IDR PUF for 2025 Q1 - QPA and Offers (As of January 21, 2026).csv',
        'air'      : '2025/Federal IDR PUF for 2025 Q1 - OON Air Ambulance (As of January 21, 2026).csv',
    },
    {
        'quarter'  : '2025Q2',
        'filetype' : 'csv',
        'oon'      : [
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk1.csv',
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk2.csv',
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk3.csv',
        ],
        'qpa'      : '2025/Federal IDR PUF for 2025 Q2 - QPA and Offers (As of January 21, 2026).csv',
        'air'      : '2025/Federal IDR PUF for 2025 Q2 - OON Air Ambulance (As of January 21, 2026).csv',
    },
]


# ── Sheet names (xlsx only) ───────────────────────────────────────────────────

SHEET_OON = 'OON Emergency and Non-Emergency'
SHEET_QPA = 'QPA and Offers'
SHEET_AIR = 'OON Air Ambulance'


# ── Normalization ─────────────────────────────────────────────────────────────

def normalize(df, quarter):
    """
    Apply all cross-quarter normalization steps to a freshly loaded dataframe.
    Safe to call on any sheet type (oon, qpa, air).
    """
    # 1. Strip whitespace from column names
    df.columns = [c.strip() for c in df.columns]

    # 2. Standardize column name variant:
    #    "as Percent of QPA" (2023 Q1-Q2) -> "as % of QPA" (2023 Q3 onward)
    df.columns = [
        c.replace('as Percent of QPA', 'as % of QPA')
         .replace('As Percent of QPA', 'as % of QPA')
        for c in df.columns
    ]

    # 3. Replace CMS suppression/missing markers with NaN across the entire dataframe.
    #    "NR" and "N/R" appear in multiple columns (Default Decision, NPI Number, etc.)
    #    and are not real values — they indicate suppressed or missing data.
    df = df.replace({'NR': pd.NA, 'N/R': pd.NA})

    # 4. Tag quarter
    df['Quarter'] = quarter

    return df


def convert_numeric_cols(df, patterns):
    """
    Convert columns matching any of the given substrings to numeric.
    Errors become NaN (coerce).
    """
    for col in df.columns:
        if any(p in col for p in patterns):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


# ── Loaders ───────────────────────────────────────────────────────────────────

def load_xlsx_sheet(filepath, sheet_name, quarter):
    print(f"  Loading xlsx: {os.path.basename(filepath)} | sheet: {sheet_name}")
    df = pd.read_excel(filepath, sheet_name=sheet_name, engine='openpyxl')
    return normalize(df, quarter)


def load_csv(filepath, quarter):
    print(f"  Loading csv:  {os.path.basename(filepath)}")
    df = pd.read_csv(filepath, low_memory=False)
    return normalize(df, quarter)


def load_csv_chunks(filepaths, quarter):
    """Load and concat multiple CSV chunks for a single quarter."""
    chunks = []
    for fp_path in filepaths:
        chunks.append(load_csv(fp_path, quarter))
    return pd.concat(chunks, ignore_index=True)


# ── Main build functions ──────────────────────────────────────────────────────

def build_oon():
    print("\n=== Building OON dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter  = q['quarter']
        filetype = q['filetype']
        source   = q['oon']

        print(f"\n{quarter}:")

        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_OON, quarter))
        else:
            all_dfs.append(load_csv_chunks([fp(f) for f in source], quarter))

    print("\nConcatenating all quarters...")
    oon = pd.concat(all_dfs, ignore_index=True)

    oon = convert_numeric_cols(oon, [
        'as % of QPA',
        'as Percent of Median',
        'NPI Number',          # adding so that nulls and strings get converted correctly
    ])

    print(f"OON total rows: {len(oon):,}")
    print(f"OON columns:    {len(oon.columns)}")
    print(f"OON quarters:   {oon['Quarter'].value_counts().sort_index().to_dict()}")
    return oon


def build_qpa():
    print("\n=== Building QPA dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter  = q['quarter']
        filetype = q['filetype']
        source   = q['qpa']

        if source is None:
            print(f"{quarter}: no QPA file — skipping")
            continue

        print(f"\n{quarter}:")

        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_QPA, quarter))
        else:
            all_dfs.append(load_csv(fp(source), quarter))

    print("\nConcatenating all quarters...")
    qpa = pd.concat(all_dfs, ignore_index=True)

    qpa = convert_numeric_cols(qpa, [
        'Amount', 'Rate', 'QPA', 'Offer', 'Payment', 'Percent'
    ])

    print(f"QPA total rows: {len(qpa):,}")
    print(f"QPA quarters:   {qpa['Quarter'].value_counts().sort_index().to_dict()}")
    return qpa


def build_air():
    print("\n=== Building Air Ambulance dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter  = q['quarter']
        filetype = q['filetype']
        source   = q['air']

        if source is None:
            print(f"{quarter}: no Air file — skipping")
            continue

        print(f"\n{quarter}:")

        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_AIR, quarter))
        else:
            all_dfs.append(load_csv(fp(source), quarter))

    print("\nConcatenating all quarters...")
    air = pd.concat(all_dfs, ignore_index=True)

    air = convert_numeric_cols(air, [
        'as % of QPA', 'Offer', 'Amount', 'Percent'
    ])

    print(f"Air total rows: {len(air):,}")
    print(f"Air quarters:   {air['Quarter'].value_counts().sort_index().to_dict()}")
    return air


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    oon = build_oon()
    qpa = build_qpa()
    air = build_air()

    # print("\n=== Saving parquet files ===")

    # oon_out = out('oon_all_quarters.parquet')
    # qpa_out = out('qpa_all_quarters.parquet')
    # air_out = out('air_all_quarters.parquet')

    # oon.to_parquet(oon_out, compression='snappy', index=False)
    # print(f"Saved: {oon_out}  ({os.path.getsize(oon_out)/1e6:.1f} MB)")

    # qpa.to_parquet(qpa_out, compression='snappy', index=False)
    # print(f"Saved: {qpa_out}  ({os.path.getsize(qpa_out)/1e6:.1f} MB)")

    # air.to_parquet(air_out, compression='snappy', index=False)
    # print(f"Saved: {air_out}  ({os.path.getsize(air_out)/1e6:.1f} MB)")

    # print("\nDone.")
    # print(f"  OON: {len(oon):,} rows")
    # print(f"  QPA: {len(qpa):,} rows")
    # print(f"  Air: {len(air):,} rows")


=== Building OON dataset ===

2023Q1:
  Loading xlsx: 2023-q1-federal-idr-puf_0.xlsx | sheet: OON Emergency and Non-Emergency

2023Q2:
  Loading xlsx: Federal-IDR-PUF-for-2023-Q2.xlsx | sheet: OON Emergency and Non-Emergency

2023Q3:
  Loading xlsx: Federal IDR PUF for 2023 Q3.xlsx | sheet: OON Emergency and Non-Emergency

2023Q4:
  Loading xlsx: Federal IDR PUF for 2023 Q4.xlsx | sheet: OON Emergency and Non-Emergency

2024Q1:
  Loading xlsx: federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q2:
  Loading xlsx: federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q3:
  Loading xlsx: federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q4:
  Loading xlsx: federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx | sheet: OON Emergency and Non-Emergency

2025Q1:
  Loading csv:  Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 

In [45]:
oon[
    oon['Provider Email Domain'].str.contains('radiology')
][[
    # 'carrier_name',
    'Provider Email Domain',
    'Carrier Name',
]].drop_duplicates()

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [46]:
oon.columns

Index(['Dispute Number', 'DLI Number', 'Payment Determination Outcome',
       'Default Decision', 'Type of Dispute', 'Provider/Facility Group Name',
       'Provider/Facility Name', 'Provider Email Domain',
       'Provider/Facility NPI Number', 'Practice/Facility Size',
       'Health Plan/Issuer Name', 'Health Plan/Issuer Email Domain',
       'Health Plan Type', 'Length of Time to Make Determination',
       'IDRE Compensation', 'Dispute Line Item Type', 'Type of Service Code',
       'Service Code', 'Place of Service Code', 'Item or Service Description',
       'Location of Service', 'Practice/Facility Specialty or Type',
       'Provider/Facility Offer as % of QPA',
       'Health Plan/Issuer Offer as % of QPA',
       'Offer Selected from Provider or Issuer',
       'Prevailing Party Offer as % of QPA', 'QPA as Percent of Median QPA',
       'Provider/Facility Offer as Percent of Median Provider/Facility Offer Amount',
       'Health Plan/Issuer Offer as Percent of Median Health

In [88]:
"""
IDR Domain Mapping
===================
Maps provider and carrier email domains to standardized group names.

Usage:
    from idr_mappings import apply_provider_map, apply_carrier_map

    oon = apply_provider_map(oon)   # adds Provider_Group, Provider_Category
    oon = apply_carrier_map(oon)    # adds Carrier_Group, Anthem_Flag
    air = apply_provider_map(air)
    air = apply_carrier_map(air)

Provider columns expected:
    'Provider Email Domain'

Carrier columns expected:
    'Health Plan/Issuer Email Domain'

Output columns added:
    Provider_Group      -- standardized provider group name (or raw domain if unmapped)
    Provider_Category   -- PE-Backed National / Physician-Owned Group /
                           Billing/Revenue Cycle / Law Firm /
                           Hospital System / Specialty Group / Unknown
    Carrier_Group       -- standardized carrier name (or raw domain if unmapped)
    Anthem_Flag         -- 'Anthem' if domain matches any Anthem pattern, else 'Other'

Category definitions:
    PE-Backed National     -- national physician staffing/management groups backed by private equity
    Physician-Owned Group  -- large national groups that are physician-owned, no PE
    Billing/Revenue Cycle  -- RCM, billing, coding, or IDR filing companies (not direct providers)
    Law Firm               -- law firms filing IDR disputes on behalf of providers
    Hospital System        -- health system entities
    Specialty Group        -- clinical specialty practices (anesthesia, radiology, neuromonitoring)
    Unknown                -- unmapped domain (raw domain value used as fallback)

Research notes on categorization decisions:
    F&A Management    : RCM firm serving freestanding ERs, moved from PE-Backed to Billing/RCM
    AGS Health        : RCM company acquired by Blackstone Aug 2025, function is RCM not clinical
    Vituity           : 100% physician-owned democratic partnership, no outside investors or PE
    Radix Health      : IDR optimization software/services, not a clinical provider
    Ventra Health     : RCM for hospital-based physicians (merger of Abeo, DuvaSawko, Gottlieb)
    Altus Health      : 27 freestanding ERs in TX, PE-backed, moved from Billing/RCM to PE-Backed
    OrthoMed Staffing : Anesthesia management group with 262 providers across 8 states
    summit-az.com     : Removed -- could not verify which IDR-filing entity this maps to
    US Revenue Cycle  : Name is self-evident, moved from Specialty Group to Billing/RCM
"""

import pandas as pd
import numpy as np
import re


# ── Provider domain map ───────────────────────────────────────────────────────

DOMAIN_MAP = {
    # PE-Backed National
    # Large, PE-backed or PE-originated national physician staffing/management groups
    "halomd.com":               ("HaloMD",                          "PE-Backed National"),
    "teamhealth.com":           ("TeamHealth",                      "PE-Backed National"),
    "envisionhealth.com":       ("Envision Healthcare",             "PE-Backed National"),
    "scp-health.com":           ("SCP Health",                      "PE-Backed National"),
    "scphealth.com":            ("SCP Health",                      "PE-Backed National"),
    "totalcare.us":             ("TotalCare",  "Specialty Group"),
    "usap.com":                 ("US Anesthesia Partners",          "PE-Backed National"),
    "usacs.com":                ("US Acute Care Solutions",         "PE-Backed National"),
    "apollomd.com":             ("Apollo MD",                       "PE-Backed National"),
    "soundphysicians.com":      ("Sound Physicians",                "PE-Backed National"),
    "altushealthsystem.com":    ("Altus Community Healthcare",      "PE-Backed National"),  # 27 freestanding ERs in TX, PE-backed at exit
    # Physician-Owned Group
    # National/large groups that are physician-owned, not PE-backed
    "vituity.com":              ("Vituity",                         "Physician-Owned Group"),  # 100% physician-owned democratic partnership, no outside investors
    # Billing / Revenue Cycle
    # Companies whose primary function is RCM, billing, coding, or IDR filing on behalf of providers
    "fam-llc.com":              ("F&A Management",                  "Billing/Revenue Cycle"),  # RCM firm serving freestanding ERs in TX
    "agshealth.com":            ("AGS Health",                      "Billing/Revenue Cycle"),  # RCM company (Blackstone-acquired Aug 2025)
    "ventrahealth.com":         ("Ventra Health",                   "Billing/Revenue Cycle"),  # RCM for hospital-based physicians (merger of Abeo, DuvaSawko, Gottlieb)
    "radixhealth.io":           ("Radix Health",                    "Billing/Revenue Cycle"),  # IDR optimization software/services, not a clinical provider
    "saparm.com":               ("Singleton Associates",            "Billing/Revenue Cycle"),
    "roundtmc.com":             ("Roundtable Medical Consultants",  "Billing/Revenue Cycle"),
    "logixhealth.com":          ("LogixHealth",                     "Billing/Revenue Cycle"),
    "r1rcm.com":                ("R1 RCM",                          "Billing/Revenue Cycle"),
    "zotecpartners.com":        ("Zotec Partners",                  "Billing/Revenue Cycle"),
    "gryphonhc.com":            ("Gryphon Healthcare",              "Billing/Revenue Cycle"),
    "qmacsmso.com":             ("QMACS MSO",                       "Billing/Revenue Cycle"),
    "mdcapitaladvisors.com":    ("MD Capital Advisors",             "Billing/Revenue Cycle"),
    "nosurprisebill.com":       ("NoSurpriseBill.com",              "Billing/Revenue Cycle"),
    "alldatahealth.com":        ("AllData Health",                  "Billing/Revenue Cycle"),
    "rightmedicalbilling.com":  ("Right Medical Billing",           "Billing/Revenue Cycle"),
    "erevenuebilling.com":      ("eRevenue / Exceptional Health Care","Billing/Revenue Cycle"),
    "usrcm.net":                ("US Revenue Cycle Mgmt",           "Billing/Revenue Cycle"),
    # Law Firms
    "callagylaw.com":           ("Callagy Law",                     "Law Firm"),
    "callagyrecovery.com":      ("Callagy Law",                     "Law Firm"),
    "gottliebandgreenspan.com": ("Gottlieb & Greenspan",            "Law Firm"),
    "halkovichlaw.com":         ("Halkovich Law",                   "Law Firm"),
    "afslaw.com":               ("AFS Law",                         "Law Firm"),
    "glynnlegal.com":           ("Glynn Legal",                     "Law Firm"),
    # Hospital Systems
    "primehealthcare.com":      ("Prime Healthcare",                "Hospital System"),
    "bmhcc.org":                ("Baptist Memorial Health",         "Hospital System"),
    "hcahealthcare.com":        ("HCA Healthcare",                  "Hospital System"),
    "commonspirit.org":         ("CommonSpirit Health",             "Hospital System"),
    "adventhealth.com":         ("AdventHealth",                    "Hospital System"),
    "wellstar.org":             ("WellStar Health",                 "Hospital System"),
    "tenethealth.com":          ("Tenet Healthcare",                "Hospital System"),
    "saintfrancis.com":         ("Saint Francis Health",            "Hospital System"),
    # Specialty Groups
    # Clinical specialty practices (anesthesia, radiology, neuromonitoring, etc.)
    "sonoranrm.com":            ("Sonoran Radiology",               "Specialty Group"),
    "mbbrm.com":                ("Mori Bean & Brooks",              "Specialty Group"),
    "empireradrm.com":          ("Empire State Radiology",          "Specialty Group"),
    "radpmg.com":               ("Houston Radiology Associates",    "Specialty Group"),
    "iairm.com":                ("Imaging Associates of Indiana",   "Specialty Group"),
    "specialtycare.net":        ("SpecialtyCare",                   "Specialty Group"),
    "orthomedstaffing.com":     ("OrthoMed Anesthesia",             "Specialty Group"),  # anesthesia management, 262 providers across 8 states
    "nmaiom.com":               ("NMA Intraoperative Monitoring",   "Specialty Group"),
    "unitedionm.com":           ("United IONM",                     "Specialty Group"),
    "pediatrix.com":            ("Pediatrix Medical Group",         "Specialty Group"),
    "anesthesiadynamics.com":   ("Anesthesia Dynamics",             "Specialty Group"),
}

# Lookup series (built once at import time for performance)
_DOMAIN_MAP_NAMES = {k: v[0] for k, v in DOMAIN_MAP.items()}
_DOMAIN_MAP_CATS  = {k: v[1] for k, v in DOMAIN_MAP.items()}


# ── Carrier domain map ────────────────────────────────────────────────────────

CARRIER_MAP = {
    "uhc.com":              "UnitedHealthcare",
    "unitedhealthcare.com": "UnitedHealthcare",
    "umr.com":              "UnitedHealthcare",
    "clearhs.com":          "ClearHealth Strategies",
    "clearhealthcloud.com": "ClearHealth Strategies",
    "aetna.com":            "Aetna",
    "aetnahealth.com":      "Aetna",
    "anthem.com":           "Anthem",
    "elevance.com":         "Anthem",
    "bcbstx.com":           "BCBS Texas",
    "hcsc.net":             "BCBS Texas",
    "cigna.com":            "Cigna",
    "evernorth.com":        "Cigna",
    "multiplan.com":        "MultiPlan/Claritev",
    "claritev.com":         "MultiPlan/Claritev",
    "bcbsfl.com":           "Florida Blue",
    "floridablue.com":      "Florida Blue",
    "bcbsil.com":           "BCBS Illinois",
    "bcbst.com":            "BCBS Tennessee",
    "bcbsaz.com":           "BCBS Arizona",
    "azblue.com":           "BCBS Arizona",
    "humana.com":           "Humana",
    "zelis.com":            "Zelis",
    "centene.com":          "Centene",
    "highmark.com":         "Highmark",
    "horizonblue.com":      "Horizon BCBS NJ",
    "bcbsnc.com":           "BCBS NC",
    "bcbssc.com":           "BCBS SC",
    "premera.com":          "Premera",
    "geha.com":             "GEHA",
    "ibx.com":              "Independence Blue Cross",
    "lablue.com":           "BCBS Louisiana",
    "bcbsm.com":            "BCBS Michigan",
    "carefirst.com":        "CareFirst BCBS",
    "bcbsok.com":           "BCBS Oklahoma",
    "bcbsks.com":           "BCBS Kansas",
    "kaiserpermanente.org": "Kaiser Permanente",
    "bcbsmn.com":           "BCBS Minnesota",
    "bcbswny.com":          "BCBS Western NY",
    "excellusbcbs.com":     "Excellus BCBS",
    "medmutual.com":        "Medical Mutual",
    "bcbs.com":             "BCBS (other)",
}


# ── Anthem regex patterns ─────────────────────────────────────────────────────
# Applied after the direct CARRIER_MAP lookup to catch Anthem-affiliated domains
# that may not be in the map (typos, subsidiaries, regional brands, TPAs)

ANTHEM_PATTERNS = [
    # Core Anthem / Elevance
    r'anthem',
    r'elevance',
    # Subsidiaries / brands
    r'amerigroup',
    r'carelon',
    r'caremore',
    r'unicare',
    r'wellpoint',
    r'healthkeepers',
    r'empireblue',
    # BCBS entities owned by Anthem
    r'bcbsga',
    # Affiliated / TPA
    r'ameriben',
]

_ANTHEM_REGEX = '|'.join(ANTHEM_PATTERNS)


# ── Mapping functions ─────────────────────────────────────────────────────────

def apply_provider_map(df, domain_col='Provider Email Domain'):
    """
    Add Provider_Group and Provider_Category columns based on email domain.
    Unmapped domains retain the raw domain value in Provider_Group
    and 'Unknown' in Provider_Category.
    """
    domain_clean = (
        df[domain_col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    df['Provider_Group']    = domain_clean.map(_DOMAIN_MAP_NAMES).fillna(domain_clean)
    df['Provider_Category'] = domain_clean.map(_DOMAIN_MAP_CATS).fillna('Unknown')

    return df


def apply_carrier_map(df, domain_col='Health Plan/Issuer Email Domain'):
    """
    Add Carrier_Group and Anthem_Flag columns based on email domain.

    Carrier_Group: direct lookup from CARRIER_MAP; unmapped domains retain
                   the raw domain value.
    Anthem_Flag:   'Anthem' if domain matches any ANTHEM_PATTERNS regex,
                   else 'Other'. Applied independently of Carrier_Group so
                   it catches Anthem-affiliated domains not in CARRIER_MAP.
    """
    domain_clean = (
        df[domain_col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # Direct carrier lookup
    df['Carrier_Group'] = domain_clean.map(CARRIER_MAP).fillna(domain_clean)

    # Anthem flag via regex (catches subsidiaries, typos, regional brands)
    df['Anthem_Flag'] = np.where(
        domain_clean.str.contains(_ANTHEM_REGEX, regex=True, na=False),
        'Anthem',
        'Other'
    )

    return df


def apply_all_maps(df,
                   provider_col='Provider Email Domain',
                   carrier_col='Health Plan/Issuer Email Domain'):
    """Convenience function — apply both maps in one call."""
    df = apply_provider_map(df, domain_col=provider_col)
    df = apply_carrier_map(df, domain_col=carrier_col)
    return df

In [89]:
oon = apply_all_maps(oon)

In [61]:
oon.columns

Index(['Dispute Number', 'DLI Number', 'Payment Determination Outcome',
       'Default Decision', 'Type of Dispute', 'Provider/Facility Group Name',
       'Provider/Facility Name', 'Provider Email Domain',
       'Provider/Facility NPI Number', 'Practice/Facility Size',
       'Health Plan/Issuer Name', 'Health Plan/Issuer Email Domain',
       'Health Plan Type', 'Length of Time to Make Determination',
       'IDRE Compensation', 'Dispute Line Item Type', 'Type of Service Code',
       'Service Code', 'Place of Service Code', 'Item or Service Description',
       'Location of Service', 'Practice/Facility Specialty or Type',
       'Provider/Facility Offer as % of QPA',
       'Health Plan/Issuer Offer as % of QPA',
       'Offer Selected from Provider or Issuer',
       'Prevailing Party Offer as % of QPA', 'QPA as Percent of Median QPA',
       'Provider/Facility Offer as Percent of Median Provider/Facility Offer Amount',
       'Health Plan/Issuer Offer as Percent of Median Health

In [97]:
# oon[oon['Provider Email Domain'].str.contains('total', na=False)][[
#     'Provider Email Domain',
#     'Provider_Group'
# ]].drop_duplicates()

oon[
    # oon['Provider Email Domain'].str.contains('tot', na=False)
    # oon['Provider Email Domain'].str.contains('halo', na=False)
    oon['Provider_Group'].str.contains('Total', na=False)
].groupby([
    # 'carrier_name',
    'Provider/Facility Name',
    'Provider/Facility NPI Number',
    'Provider/Facility Group Name',
    'Provider Email Domain',
    'Provider_Group',
]).agg(
    dispute_count=('Dispute Number', 'nunique')
).reset_index().sort_values(
    'dispute_count',
    ascending=False
)

In [98]:
def add_outcome_bucket(df,
                       outcome_col='Payment Determination Outcome',
                       default_col='Default Decision'):
    """
    Add Outcome_Bucket column with four mutually exclusive categories:
        Contested Provider Win
        Contested Plan Win
        Default Provider Win
        Default Plan Win
        Other / Split
 
    Requires both 'Payment Determination Outcome' and 'Default Decision' columns.
    """
    is_default  = df[default_col] == 'Yes'
    is_prov_win = df[outcome_col].str.contains('Provider', na=False)
    is_plan_win = df[outcome_col].str.contains('Plan|Issuer', na=False)
 
    conditions = [
        (~is_default) & is_prov_win,
        (~is_default) & is_plan_win,
        is_default    & is_prov_win,
        is_default    & is_plan_win,
    ]
    choices = [
        'Contested Provider Win',
        'Contested Plan Win',
        'Default Provider Win',
        'Default Plan Win',
    ]
 
    import numpy as np
    df['Outcome_Bucket'] = np.select(conditions, choices, default='Other / Split')
 
    return df

oon = add_outcome_bucket(oon)

In [99]:
# oon[oon['Provider Email Domain'].str.contains('total', na=False)][[
#     'Provider Email Domain',
#     'Provider_Group'
# ]].drop_duplicates()

oon.groupby([
    'Outcome_Bucket',
]).agg(
    dispute_count=('Dispute Number', 'nunique')
).reset_index().sort_values(
    'dispute_count',
    ascending=False
)

,Outcome_Bucket,dispute_count
1,Contested Provider Win,1541801
3,Default Provider Win,406838
0,Contested Plan Win,253350
2,Default Plan Win,68052
4,Other / Split,1560


In [101]:
def clean_currency_cols(df):
    """Strip $ and commas from any object columns that look like currency."""
    for col in df.select_dtypes(include='object').columns:
        # Sample non-null values
        sample = df[col].dropna().head(100)
        # If any look like $X.XX or $X,XXX convert the whole column
        if sample.astype(str).str.match(r'^\$[\d,]+\.?\d*$').any():
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
                errors='coerce'
            )
    return df

oon = clean_currency_cols(oon)
qpa = clean_currency_cols(qpa)
air = clean_currency_cols(air)

In [103]:
oon['IDRE Compensation'] = pd.to_numeric(
    oon['IDRE Compensation'].astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
    errors='coerce'
)

In [105]:
code_cols = ['Service Code', 'Place of Service Code', 'Type of Service Code']
for col in code_cols:
    if col in oon.columns:
        oon[col] = oon[col].astype(str).replace('nan', pd.NA)

In [107]:
for col in oon.select_dtypes(include='object').columns:
    unique_types = oon[col].dropna().map(type).unique()
    if len(unique_types) > 1:
        print(f'{col}: {unique_types}')

Item or Service Description: [<class 'str'> <class 'int'>]
Practice/Facility Specialty or Type: [<class 'str'> <class 'int'>]


In [109]:
str_cols = [
    'Service Code',
    'Place of Service Code', 
    'Type of Service Code',
    'Item or Service Description',
    'Practice/Facility Specialty or Type',
]
for col in str_cols:
    if col in oon.columns:
        oon[col] = oon[col].astype(str).replace('nan', pd.NA)

In [120]:
for col in qpa.select_dtypes(include='object').columns:
    unique_types = qpa[col].dropna().map(type).unique()
    if len(unique_types) > 1:
        print(f'{col}: {unique_types}')

Service Code: [<class 'str'> <class 'int'> <class 'float'>]
Place of Service Code: [<class 'float'> <class 'int'> <class 'str'>]


In [121]:
str_cols = [
    'Service Code',
    'Place of Service Code', 
]
for col in str_cols:
    if col in qpa.columns:
        qpa[col] = qpa[col].astype(str).replace('nan', pd.NA)

In [115]:
for col in air.select_dtypes(include='object').columns:
    unique_types = air[col].dropna().map(type).unique()
    if len(unique_types) > 1:
        print(f'{col}: {unique_types}')

Provider/Facility NPI Number: [<class 'str'> <class 'int'>]
IDRE Compensation: [<class 'int'> <class 'float'> <class 'str'>]
Service Code: [<class 'str'> <class 'int'>]
Place of Service Code: [<class 'int'> <class 'str'>]
Ambulance Vehicle Type: [<class 'str'> <class 'int'>]
Ambulance Vehicle Clinical Capacity Level: [<class 'str'> <class 'int'>]
Air Ambulance Vehicle Clinical Capacity Level: [<class 'str'> <class 'int'>]


In [119]:
qpa = build_qpa()


=== Building QPA dataset ===

2023Q1:
  Loading xlsx: 2023-q1-federal-idr-puf_0.xlsx | sheet: QPA and Offers

2023Q2:
  Loading xlsx: Federal-IDR-PUF-for-2023-Q2.xlsx | sheet: QPA and Offers

2023Q3:
  Loading xlsx: Federal IDR PUF for 2023 Q3.xlsx | sheet: QPA and Offers

2023Q4:
  Loading xlsx: Federal IDR PUF for 2023 Q4.xlsx | sheet: QPA and Offers

2024Q1:
  Loading xlsx: federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx | sheet: QPA and Offers

2024Q2:
  Loading xlsx: federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx | sheet: QPA and Offers

2024Q3:
  Loading xlsx: federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx | sheet: QPA and Offers

2024Q4:
  Loading xlsx: federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx | sheet: QPA and Offers

2025Q1:
  Loading csv:  Federal IDR PUF for 2025 Q1 - QPA and Offers (As of January 21, 2026).csv

2025Q2:
  Loading csv:  Federal IDR PUF for 2025 Q2 - QPA and Offers (As of January 21, 2026).csv

Concatenating all quarters...
QPA total r

In [128]:
qpa

,Service Code,Place of Service Code,Geographical Region,QPA,Provider/Facility Offer,Health Plan/Issuer Offer,Offer Selected from Provider or Issuer,Prevailing Offer,Default Decision,Quarter,Initiating Party,Dispute Line Item Type,Type of Service Code,Air Ambulance Pick-up Location
0,A0436,42.0,"Aberdeen, WA",NaN,NaN,NaN,NaN,NaN,No,2023Q1,NaN,NaN,NaN,NaN
1,A0431,42.0,"Aberdeen, WA",NaN,NaN,NaN,NaN,NaN,No,2023Q1,NaN,NaN,NaN,NaN
2,A0431,42.0,"Aberdeen, WA",NaN,NaN,NaN,NaN,NaN,No,2023Q1,NaN,NaN,NaN,NaN
3,A0431,42.0,"Aberdeen, WA",NaN,NaN,NaN,NaN,NaN,No,2023Q1,NaN,NaN,NaN,NaN
4,A0431,42.0,"Aberdeen, WA",NaN,NaN,NaN,NaN,NaN,No,2023Q1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5330348,95861,22,"Zanesville, OH",NaN,NaN,NaN,NaN,NaN,No,2025Q2,Health care provider,Single,CPT,NaN
5330349,95938,21,"Zanesville, OH",NaN,NaN,NaN,NaN,NaN,Yes,2025Q2,Health care provider,Single,CPT,NaN
5330350,99284,23,"Zanesville, OH",NaN,NaN,NaN,NaN,NaN,No,2025Q2,Health care provider,Batched,CPT,NaN
5330351,73562,22,"Zapata, TX",NaN,NaN,NaN,NaN,NaN,No,2025Q2,Health care provider,Batched,CPT,NaN


In [117]:
str_cols = [
    'Provider/Facility NPI Number',
    'IDRE Compensation', 
    'Service Code',
    'Place of Service Code',
    'Ambulance Vehicle Type',
    'Ambulance Vehicle Clinical Capacity Level',
    'Air Ambulance Vehicle Clinical Capacity Level'
]
for col in str_cols:
    if col in air.columns:
        air[col] = air[col].astype(str).replace('nan', pd.NA)

In [114]:
print("\n=== Saving parquet files ===")

oon_out = out('oon_all_quarters.parquet')
qpa_out = out('qpa_all_quarters.parquet')
air_out = out('air_all_quarters.parquet')

oon.to_parquet(oon_out, compression='snappy', index=False)
print(f"Saved: {oon_out}  ({os.path.getsize(oon_out)/1e6:.1f} MB)")

qpa.to_parquet(qpa_out, compression='snappy', index=False)
print(f"Saved: {qpa_out}  ({os.path.getsize(qpa_out)/1e6:.1f} MB)")

air.to_parquet(air_out, compression='snappy', index=False)
print(f"Saved: {air_out}  ({os.path.getsize(air_out)/1e6:.1f} MB)")

print("\nDone.")
print(f"  OON: {len(oon):,} rows")
print(f"  QPA: {len(qpa):,} rows")
print(f"  Air: {len(air):,} rows")


=== Saving parquet files ===
Saved: ./parquet\oon_all_quarters.parquet  (195.4 MB)
Saved: ./parquet\qpa_all_quarters.parquet  (41.1 MB)


ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column Provider/Facility NPI Number with type object')

In [34]:
"""
IDR Supplemental Tables Loader
================================
Loads all 10 quarters of CMS Federal IDR Supplemental Tables and produces
one parquet per logical sub-table, each with a Quarter column appended.

Output parquet files (saved to OUT_DIR):
  supp_summary_disputes_initiated.parquet     -- Table 1  (disputes initiated by type)
  supp_provider_size.parquet                  -- Table 2  (disputes by provider size)
  supp_plan_type.parquet                      -- Table 3  (disputes by health plan type)
  supp_closure_reasons.parquet                -- Table 4  (closure reasons)
  supp_eligibility.parquet                    -- Table 5  (eligibility determinations)
  supp_initiations_by_state.parquet           -- Table 7  (initiations by state)
  supp_top_initiating_parties.parquet         -- Table 8  (top 10 initiating parties)
  supp_top_noninitiating_parties.parquet      -- Table 9  (top 10 non-initiating parties)
  supp_payment_determination_outcomes.parquet -- Table 12 (payment determination outcomes)
  supp_qpa_by_cost_band.parquet               -- Table 13 (prevailing offer vs QPA by cost band)
  supp_qpa_by_specialty.parquet               -- Table 14 (prevailing offer vs QPA by specialty)

Notes:
  - All sheets are human-readable stacked tables, not raw tabular data
  - Each sub-table is parsed by detecting its title row
  - 2025 Q1 adds a 'Church Plan' column to Table 9 (non-initiating parties)
    — earlier quarters will have NaN for that column after concat
  - Table numbering is consistent across all 10 quarters
  - 'Contents' sheet is skipped (legend only)
  - Some cells contain comma-formatted strings e.g. '63,395' — to_num handles these
"""

import pandas as pd
import numpy as np
import os
import re

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = r"."
OUT_DIR  = r"./parquet"

def fp(f):  return os.path.join(DATA_DIR, f)
def out(f): return os.path.join(OUT_DIR, f)


# ── File manifest ─────────────────────────────────────────────────────────────
QUARTERS = [
    ('2023Q1', '2023/Federal-IDR-Supplemental-Tables-for-2023-Q1.xlsx'),
    ('2023Q2', '2023/Federal-IDR-Supplemental-Tables-for-2023-Q2.xlsx'),
    ('2023Q3', '2023/Federal IDR Supplemental Tables for 2023 Q3.xlsx'),
    ('2023Q4', '2023/Federal IDR Supplemental Tables for 2023 Q4.xlsx'),
    ('2024Q1', '2024/federal-idr-supplemental-tables-for-2024-q1-as-of-march-18-2025.xlsx'),
    ('2024Q2', '2024/federal-idr-supplemental-tables-for-2024-q2-as-of-march-18-2025.xlsx'),
    ('2024Q3', '2024/federal-idr-supplemental-tables-for-2024-q3-as-of-may-28-2025_0.xlsx'),
    ('2024Q4', '2024/federal-idr-supplemental-tables-for-2024-q4-as-of-may-28-2025-1.xlsx'),
    ('2025Q1', '2025/federal-idr-supplemental-tables-2025-q1.xlsx'),
    ('2025Q2', '2025/federal-idr-supplemental-tables-2025-q2.xlsx'),
]

# Note: 2024 files have .xslx typo in some filenames on disk —
# update above if needed to match your actual filenames exactly.


# ── Numeric conversion helper ─────────────────────────────────────────────────

def to_num(series):
    """
    Strip comma formatting (e.g. '63,395') then convert to numeric.
    Handles both string and already-numeric inputs safely.
    Unconvertible values become NaN.
    """
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )


# ── Core parser ───────────────────────────────────────────────────────────────

def load_raw(filepath, sheet_name):
    """Load a sheet with no header parsing — returns raw dataframe."""
    return pd.read_excel(
        filepath, sheet_name=sheet_name,
        header=None, engine='openpyxl'
    )


def find_table_start(raw, table_number):
    """
    Find the row index where 'Table {N}:' appears in column 0.
    Returns None if not found.
    """
    pattern = re.compile(rf'Table\s+{table_number}\s*:', re.IGNORECASE)
    for i, val in enumerate(raw[0]):
        if isinstance(val, str) and pattern.match(val.strip()):
            return i
    return None


def extract_simple_table(raw, table_number, header_row_offset=1, skip_rows=0):
    """
    Extract a sub-table starting at Table N.
    header_row_offset: rows after the title row where headers live
    skip_rows: additional rows to skip after headers before data starts
    Reads until the next blank row or next Table title.
    """
    start = find_table_start(raw, table_number)
    if start is None:
        return None

    header_row = start + header_row_offset
    data_start  = header_row + 1 + skip_rows

    # Find end: next Table title or end of data
    end = len(raw)
    for i in range(start + 1, len(raw)):
        val = raw.iloc[i, 0]
        if isinstance(val, str) and re.match(r'Table\s+\d+\s*:', val.strip(), re.IGNORECASE):
            end = i
            break

    headers = raw.iloc[header_row].tolist()
    data    = raw.iloc[data_start:end].copy()

    # Drop fully blank rows
    data = data.dropna(how='all')

    data.columns = headers
    data = data.reset_index(drop=True)

    # Drop fully blank columns
    data = data.dropna(axis=1, how='all')

    return data


# ── Sheet-specific parsers ────────────────────────────────────────────────────

def parse_summary_tables(raw, quarter):
    """
    Summary Tables sheet — parse each sub-table individually.
    Returns dict of {output_name: dataframe}
    """
    results = {}

    # Table 1: Disputes Initiated (3 cols: OON, Air, Total)
    t1 = extract_simple_table(raw, 1, header_row_offset=1)
    if t1 is not None:
        t1.columns = ['Category', 'OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']
        t1 = t1[t1['Category'].notna()].copy()
        for col in ['OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']:
            t1[col] = to_num(t1[col])
        t1['Quarter'] = quarter
        results['summary_disputes_initiated'] = t1

    # Table 2: Provider/Facility Size
    t2 = extract_simple_table(raw, 2, header_row_offset=1)
    if t2 is not None:
        t2.columns = ['Provider_Size', 'OON_Emergency_NonEmergency']
        t2 = t2[t2['Provider_Size'].notna()].copy()
        t2['OON_Emergency_NonEmergency'] = to_num(t2['OON_Emergency_NonEmergency'])
        t2['Quarter'] = quarter
        results['provider_size'] = t2

    # Table 3: Health Plan Type
    t3 = extract_simple_table(raw, 3, header_row_offset=1)
    if t3 is not None:
        t3.columns = ['Health_Plan_Type', 'OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']
        t3 = t3[t3['Health_Plan_Type'].notna()].copy()
        for col in ['OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']:
            t3[col] = to_num(t3[col])
        t3['Quarter'] = quarter
        results['plan_type'] = t3

    # Table 4: Closure Reasons
    t4 = extract_simple_table(raw, 4, header_row_offset=1)
    if t4 is not None:
        t4 = t4[t4.iloc[:, 0].notna()].copy()
        # Convert all non-first columns to numeric
        for col in t4.columns[1:]:
            t4[col] = to_num(t4[col])
        t4['Quarter'] = quarter
        results['closure_reasons'] = t4

    # Table 5: Eligibility
    t5 = extract_simple_table(raw, 5, header_row_offset=1)
    if t5 is not None:
        t5 = t5[t5.iloc[:, 0].notna()].copy()
        for col in t5.columns[1:]:
            t5[col] = to_num(t5[col])
        t5['Quarter'] = quarter
        results['eligibility'] = t5

    return results


def parse_initiations_by_state(raw, quarter):
    """Table 7 — state initiations."""
    start = find_table_start(raw, 7)
    if start is None:
        return None

    headers = ['State_Territory', 'OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']

    data_start = start + 5
    data = raw.iloc[data_start:].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    rows = []
    for _, row in data.iterrows():
        val = row.iloc[0]
        if isinstance(val, str) and re.match(r'Table\s+\d+', val.strip(), re.IGNORECASE):
            break
        rows.append(row.tolist()[:4])

    df = pd.DataFrame(rows, columns=headers)
    df['State_Territory'] = df['State_Territory'].astype(str).str.replace(r'[+*†‡]', '', regex=True).str.strip()
    for col in ['OON_Emergency_NonEmergency', 'Air_Ambulance', 'Total']:
        df[col] = to_num(df[col])
    df['Quarter'] = quarter
    return df


def parse_top_initiating(raw, quarter):
    """Table 8 — top 10 initiating parties."""
    start = find_table_start(raw, 8)
    if start is None:
        return None

    data_start = start + 2
    end = find_table_start(raw, 9)
    if end is None:
        end = len(raw)

    data = raw.iloc[data_start:end].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    df = pd.DataFrame({
        'Initiating_Party': data.iloc[:, 0].values,
        'Total_Disputes':   to_num(data.iloc[:, 1]),
        'Pct_of_Disputes':  to_num(data.iloc[:, 2]),
    })
    df['Quarter'] = quarter
    return df


def parse_top_noninitiating(raw, quarter):
    """
    Table 9 — top 10 non-initiating parties with plan type subtotals.
    2025 Q1+ adds a 'Church Plan' column — handled by position.
    """
    start = find_table_start(raw, 9)
    if start is None:
        return None

    data_start = start + 3
    end = find_table_start(raw, 10)
    if end is None:
        end = len(raw)

    data = raw.iloc[data_start:end].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    base_cols = [
        'Non_Initiating_Party',
        'Total_Disputes',
        'Pct_of_Disputes',
        'Self_Insured',
        'Fully_Insured',
        'Individual_Market',
        'FEHB',
        'Non_Federal_Govt',
    ]

    n_cols = data.shape[1]
    if n_cols >= 10:
        cols = base_cols + ['Church_Plan', 'No_Issuer_Response']
    else:
        cols = base_cols + ['No_Issuer_Response']

    data_trimmed = data.iloc[:, :len(cols)].copy()
    data_trimmed.columns = cols

    for c in cols[1:]:
        data_trimmed[c] = to_num(data_trimmed[c])

    data_trimmed['Quarter'] = quarter
    return data_trimmed.reset_index(drop=True)


def parse_payment_determination_outcomes(raw, quarter):
    """Table 12 — payment determination outcome counts."""
    start = find_table_start(raw, 12)
    if start is None:
        return None

    data_start = start + 2
    end = find_table_start(raw, 13)
    if end is None:
        end = len(raw)

    data = raw.iloc[data_start:end].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    df = pd.DataFrame({
        'Metric':           data.iloc[:, 0].values,
        'OON_NonEmergency': to_num(data.iloc[:, 1]),
        'Air_Ambulance':    to_num(data.iloc[:, 2]),
        'Total':            to_num(data.iloc[:, 3]),
    })
    df['Quarter'] = quarter
    return df


def parse_qpa_by_cost_band(raw, quarter):
    """Table 13 — prevailing offer vs QPA by cost band."""
    start = find_table_start(raw, 13)
    if start is None:
        return None

    data_start = start + 2
    end = find_table_start(raw, 14)
    if end is None:
        end = len(raw)

    data = raw.iloc[data_start:end].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    df = pd.DataFrame({
        'QPA_Range':                       data.iloc[:, 0].values,
        'Median_Prevailing_Offer_Pct_QPA': to_num(data.iloc[:, 1]),
        'Total_Payment_Determinations':    to_num(data.iloc[:, 2]),
        'Total_Items_or_Services':         to_num(data.iloc[:, 3]),
    })
    df['Quarter'] = quarter
    return df


def parse_qpa_by_specialty(raw, quarter):
    """Table 14 — prevailing offer vs QPA by specialty."""
    start = find_table_start(raw, 14)
    if start is None:
        return None

    data_start = start + 2
    data = raw.iloc[data_start:].copy()
    data = data.dropna(how='all')
    data = data[data.iloc[:, 0].notna()]

    df = pd.DataFrame({
        'CPT_Range':                       data.iloc[:, 0].values,
        'Specialty':                       data.iloc[:, 1].values,
        'Median_Prevailing_Offer_Pct_QPA': to_num(data.iloc[:, 2]),
        'Total_Payment_Determinations':    to_num(data.iloc[:, 3]),
        'Total_Items_or_Services':         to_num(data.iloc[:, 4]),
    })
    df['Quarter'] = quarter
    return df


# ── Main ──────────────────────────────────────────────────────────────────────

def build_supplemental():
    os.makedirs(OUT_DIR, exist_ok=True)

    acc = {
        'summary_disputes_initiated':       [],
        'provider_size':                    [],
        'plan_type':                        [],
        'closure_reasons':                  [],
        'eligibility':                      [],
        'initiations_by_state':             [],
        'top_initiating_parties':           [],
        'top_noninitiating_parties':        [],
        'payment_determination_outcomes':   [],
        'qpa_by_cost_band':                 [],
        'qpa_by_specialty':                 [],
    }

    for quarter, filename in QUARTERS:
        path = fp(filename)
        print(f'\n{quarter}: {os.path.basename(path)}')

        raw_summary = load_raw(path, 'Summary Tables')
        summary_results = parse_summary_tables(raw_summary, quarter)
        for key, df in summary_results.items():
            if df is not None and len(df) > 0:
                acc[key].append(df)
                print(f'  Summary/{key}: {len(df)} rows')

        raw_state = load_raw(path, 'Initiations by State')
        df_state = parse_initiations_by_state(raw_state, quarter)
        if df_state is not None:
            acc['initiations_by_state'].append(df_state)
            print(f'  Initiations by State: {len(df_state)} rows')

        raw_parties = load_raw(path, 'Top Disputing Parties')
        df_init = parse_top_initiating(raw_parties, quarter)
        if df_init is not None:
            acc['top_initiating_parties'].append(df_init)
            print(f'  Top Initiating: {len(df_init)} rows')

        df_noninit = parse_top_noninitiating(raw_parties, quarter)
        if df_noninit is not None:
            acc['top_noninitiating_parties'].append(df_noninit)
            print(f'  Top Non-Initiating: {len(df_noninit)} rows')

        raw_outcomes = load_raw(path, 'Payment Determination Outcomes')
        df_outcomes = parse_payment_determination_outcomes(raw_outcomes, quarter)
        if df_outcomes is not None:
            acc['payment_determination_outcomes'].append(df_outcomes)
            print(f'  Payment Outcomes: {len(df_outcomes)} rows')

        df_cost = parse_qpa_by_cost_band(raw_outcomes, quarter)
        if df_cost is not None:
            acc['qpa_by_cost_band'].append(df_cost)
            print(f'  QPA by Cost Band: {len(df_cost)} rows')

        df_spec = parse_qpa_by_specialty(raw_outcomes, quarter)
        if df_spec is not None:
            acc['qpa_by_specialty'].append(df_spec)
            print(f'  QPA by Specialty: {len(df_spec)} rows')

    print('\n=== Saving parquet files ===')
    for key, dfs in acc.items():
        if not dfs:
            print(f'  SKIPPED (no data): {key}')
            continue
        combined = pd.concat(dfs, ignore_index=True)
        out_path = out(f'supp_{key}.parquet')
        combined.to_parquet(out_path, compression='snappy', index=False)
        print(f'  {key}: {len(combined)} rows -> {out_path}')


if __name__ == '__main__':
    build_supplemental()


2023Q1: Federal-IDR-Supplemental-Tables-for-2023-Q1.xlsx
  Summary/summary_disputes_initiated: 4 rows
  Summary/provider_size: 7 rows
  Summary/plan_type: 8 rows
  Summary/closure_reasons: 2 rows
  Summary/eligibility: 4 rows
  Initiations by State: 58 rows
  Top Initiating: 10 rows
  Top Non-Initiating: 10 rows
  Payment Outcomes: 6 rows
  QPA by Cost Band: 6 rows
  QPA by Specialty: 14 rows

2023Q2: Federal-IDR-Supplemental-Tables-for-2023-Q2.xlsx
  Summary/summary_disputes_initiated: 4 rows
  Summary/provider_size: 7 rows
  Summary/plan_type: 8 rows
  Summary/closure_reasons: 2 rows
  Summary/eligibility: 4 rows
  Initiations by State: 58 rows
  Top Initiating: 10 rows
  Top Non-Initiating: 10 rows
  Payment Outcomes: 6 rows
  QPA by Cost Band: 6 rows
  QPA by Specialty: 14 rows

2023Q3: Federal IDR Supplemental Tables for 2023 Q3.xlsx
  Summary/summary_disputes_initiated: 4 rows
  Summary/provider_size: 7 rows
  Summary/plan_type: 8 rows
  Summary/closure_reasons: 2 rows
  Summary

In [39]:
len(oon)

TypeError: object of type 'NoneType' has no len()

In [37]:
"""
IDR Parquet Loader
===================
Loads all IDR parquet files into named dataframes.

PUF dataframes:
  oon  -- OON Emergency & Non-Emergency (all quarters)
  qpa  -- QPA and Offers (all quarters)
  air  -- OON Air Ambulance (all quarters)

Supplemental dataframes:
  supp_disputes_initiated      -- Table 1  disputes initiated by type
  supp_provider_size           -- Table 2  disputes by provider size
  supp_plan_type               -- Table 3  disputes by health plan type
  supp_closure_reasons         -- Table 4  closure reasons
  supp_eligibility             -- Table 5  eligibility determinations
  supp_state                   -- Table 7  initiations by state
  supp_top_initiating          -- Table 8  top 10 initiating parties
  supp_top_noninitiating       -- Table 9  top 10 non-initiating parties
  supp_outcomes                -- Table 12 payment determination outcomes
  supp_cost_band               -- Table 13 prevailing offer vs QPA by cost band
  supp_specialty               -- Table 14 prevailing offer vs QPA by specialty
"""

import pandas as pd
from pathlib import Path

# ── Update to wherever your parquet files live ────────────────────────────────
PARQUET_DIR = Path("./parquet")

def pq(filename):
    return PARQUET_DIR / filename


def load_all(include_puf=True):

    if include_puf:
        print("Loading PUF files...")

        oon = pd.read_parquet(pq("oon_all_quarters.parquet"))
        print(f"  oon:  {len(oon):>12,} rows  |  {oon['Quarter'].nunique()} quarters  |  {len(oon.columns)} columns")

        qpa = pd.read_parquet(pq("qpa_all_quarters.parquet"))
        print(f"  qpa:  {len(qpa):>12,} rows  |  {qpa['Quarter'].nunique()} quarters  |  {len(qpa.columns)} columns")

        air = pd.read_parquet(pq("air_all_quarters.parquet"))
        print(f"  air:  {len(air):>12,} rows  |  {air['Quarter'].nunique()} quarters  |  {len(air.columns)} columns")

    else:
        print("Skipping PUF files.")
        oon = qpa = air = None

    print("\nLoading supplemental tables...")

    supp_disputes_initiated = pd.read_parquet(pq("supp_summary_disputes_initiated.parquet"))
    print(f"  supp_disputes_initiated:   {len(supp_disputes_initiated):>6,} rows")

    supp_provider_size = pd.read_parquet(pq("supp_provider_size.parquet"))
    print(f"  supp_provider_size:        {len(supp_provider_size):>6,} rows")

    supp_plan_type = pd.read_parquet(pq("supp_plan_type.parquet"))
    print(f"  supp_plan_type:            {len(supp_plan_type):>6,} rows")

    supp_closure_reasons = pd.read_parquet(pq("supp_closure_reasons.parquet"))
    print(f"  supp_closure_reasons:      {len(supp_closure_reasons):>6,} rows")

    supp_eligibility = pd.read_parquet(pq("supp_eligibility.parquet"))
    print(f"  supp_eligibility:          {len(supp_eligibility):>6,} rows")

    supp_state = pd.read_parquet(pq("supp_initiations_by_state.parquet"))
    print(f"  supp_state:                {len(supp_state):>6,} rows")

    supp_top_initiating = pd.read_parquet(pq("supp_top_initiating_parties.parquet"))
    print(f"  supp_top_initiating:       {len(supp_top_initiating):>6,} rows")

    supp_top_noninitiating = pd.read_parquet(pq("supp_top_noninitiating_parties.parquet"))
    print(f"  supp_top_noninitiating:    {len(supp_top_noninitiating):>6,} rows")

    supp_outcomes = pd.read_parquet(pq("supp_payment_determination_outcomes.parquet"))
    print(f"  supp_outcomes:             {len(supp_outcomes):>6,} rows")

    supp_cost_band = pd.read_parquet(pq("supp_qpa_by_cost_band.parquet"))
    print(f"  supp_cost_band:            {len(supp_cost_band):>6,} rows")

    supp_specialty = pd.read_parquet(pq("supp_qpa_by_specialty.parquet"))
    print(f"  supp_specialty:            {len(supp_specialty):>6,} rows")

    print("\nAll files loaded.")

    return (
        oon, qpa, air,
        supp_disputes_initiated,
        supp_provider_size,
        supp_plan_type,
        supp_closure_reasons,
        supp_eligibility,
        supp_state,
        supp_top_initiating,
        supp_top_noninitiating,
        supp_outcomes,
        supp_cost_band,
        supp_specialty,
    )


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    (
        oon, qpa, air,
        supp_disputes_initiated,
        supp_provider_size,
        supp_plan_type,
        supp_closure_reasons,
        supp_eligibility,
        supp_state,
        supp_top_initiating,
        supp_top_noninitiating,
        supp_outcomes,
        supp_cost_band,
        supp_specialty,
    ) = load_all(include_puf=False)  # NOTE: If you want to include/exclude PUF files, change this

Skipping PUF files.

Loading supplemental tables...
  supp_disputes_initiated:       40 rows
  supp_provider_size:            70 rows
  supp_plan_type:                80 rows
  supp_closure_reasons:          20 rows
  supp_eligibility:              40 rows
  supp_state:                   576 rows
  supp_top_initiating:          100 rows
  supp_top_noninitiating:       100 rows
  supp_outcomes:                 61 rows
  supp_cost_band:                60 rows
  supp_specialty:               238 rows

All files loaded.


In [129]:
"""
IDR PUF Data Loader — Final
============================
Loads all quarters (2023 Q1 - 2025 Q2) of CMS Federal IDR Public Use Files
into three consolidated dataframes:
  - oon : OON Emergency & Non-Emergency
  - qpa : QPA and Offers
  - air : OON Air Ambulance

Output files (parquet):
  - oon_all_quarters.parquet
  - qpa_all_quarters.parquet
  - air_all_quarters.parquet

Transformations applied:
  - Column rename: "as Percent of QPA" -> "as % of QPA" (changed in 2023 Q3)
  - Global replace: "NR" / "N/R" -> NaN (CMS suppression markers)
  - Currency columns: strip "$" and "," before numeric conversion (IDRE Compensation)
  - Force string columns: Service Code, Place of Service Code, Type of Service Code,
      Item or Service Description, Practice/Facility Specialty or Type,
      Ambulance Vehicle Type, Ambulance Vehicle Clinical Capacity Level
  - Numeric columns: offer/QPA percent columns, NPI Number, Length of Time,
      IDRE Compensation, QPA dollar amounts
  - Provider domain mapping: Provider_Group, Provider_Category (from idr_mappings.py)
  - Carrier domain mapping: Carrier_Group, Anthem_Flag (from idr_mappings.py)
  - Outcome bucket: Outcome_Bucket (OON and Air only)
  - Quarter tag: Quarter column added to every row

Notes:
  - 2025 files are CSVs; 2023/2024 are xlsx
  - 2025 Q1 OON split into 2 chunks; 2025 Q2 OON split into 3 chunks
  - TODO: Remove leading space from QPA and Offers sheetname in 2023Q2 file on fresh download
      > same file, rename 'Air Ambulance' sheet to 'OON Air Ambulance'
"""

import pandas as pd
import numpy as np
import os
from idr_mappings import apply_all_maps, add_outcome_bucket

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = r"."
OUT_DIR  = r"./parquet"
os.makedirs(OUT_DIR, exist_ok=True)

def fp(filename):  return os.path.join(DATA_DIR, filename)
def out(filename): return os.path.join(OUT_DIR, filename)


# ── File manifest ─────────────────────────────────────────────────────────────
QUARTERS = [
    {
        'quarter'  : '2023Q1',
        'filetype' : 'xlsx',
        'oon'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
        'qpa'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
        'air'      : '2023/2023-q1-federal-idr-puf_0.xlsx',
    },
    {
        'quarter'  : '2023Q2',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
        'qpa'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
        'air'      : '2023/Federal-IDR-PUF-for-2023-Q2.xlsx',
    },
    {
        'quarter'  : '2023Q3',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
        'qpa'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
        'air'      : '2023/Federal IDR PUF for 2023 Q3.xlsx',
    },
    {
        'quarter'  : '2023Q4',
        'filetype' : 'xlsx',
        'oon'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
        'qpa'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
        'air'      : '2023/Federal IDR PUF for 2023 Q4.xlsx',
    },
    {
        'quarter'  : '2024Q1',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx',
    },
    {
        'quarter'  : '2024Q2',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx',
    },
    {
        'quarter'  : '2024Q3',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx',
    },
    {
        'quarter'  : '2024Q4',
        'filetype' : 'xlsx',
        'oon'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
        'qpa'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
        'air'      : '2024/federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx',
    },
    {
        'quarter'  : '2025Q1',
        'filetype' : 'csv',
        'oon'      : [
            '2025/Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk1.csv',
            '2025/Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk2.csv',
        ],
        'qpa'      : '2025/Federal IDR PUF for 2025 Q1 - QPA and Offers (As of January 21, 2026).csv',
        'air'      : '2025/Federal IDR PUF for 2025 Q1 - OON Air Ambulance (As of January 21, 2026).csv',
    },
    {
        'quarter'  : '2025Q2',
        'filetype' : 'csv',
        'oon'      : [
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk1.csv',
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk2.csv',
            '2025/Federal IDR PUF for 2025 Q2 - OON Emergency & Non-Emergency (As of January 21, 2026) chunk3.csv',
        ],
        'qpa'      : '2025/Federal IDR PUF for 2025 Q2 - QPA and Offers (As of January 21, 2026).csv',
        'air'      : '2025/Federal IDR PUF for 2025 Q2 - OON Air Ambulance (As of January 21, 2026).csv',
    },
]

SHEET_OON = 'OON Emergency and Non-Emergency'
SHEET_QPA = 'QPA and Offers'
SHEET_AIR = 'OON Air Ambulance'


# ── Type conversion helpers ───────────────────────────────────────────────────

# Columns that must always be string (codes, descriptions, names)
# These can come through as int from xlsx in some quarters
FORCE_STR_COLS = [
    'Service Code',
    'Place of Service Code',
    'Type of Service Code',
    'Item or Service Description',
    'Practice/Facility Specialty or Type',
    'Air Ambulance Vehicle Type',                    
    'Air Ambulance Vehicle Clinical Capacity Level',  
    'Geographical Region',
]

# Columns to convert to numeric after stripping currency formatting
CURRENCY_COLS = [
    'IDRE Compensation',    # stored as '$0.00' in some quarters
    'QPA',                  # qpa sheet — dollar amounts
    'Provider/Facility Offer',
    'Health Plan/Issuer Offer',
    'Prevailing Offer',
]

# Patterns for offer/percentage columns to convert to numeric
NUMERIC_PATTERNS = [
    'as % of QPA',
    'as Percent of Median',
    'NPI Number',
    'Length of Time',
    'IDRE Compensation',
]


def convert_numeric_cols(df, patterns):
    """Convert columns matching any substring pattern to numeric."""
    for col in df.columns:
        if any(p in col for p in patterns):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def convert_currency_cols(df):
    """Strip $ and commas then convert to numeric for currency-formatted columns."""
    for col in CURRENCY_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str)
                       .str.replace(r'[$,]', '', regex=True)
                       .str.strip(),
                errors='coerce'
            )
    return df


def force_string_cols(df):
    """
    Cast code/description columns to string.
    These come through as int in some xlsx quarters and str in others.
    Null values become pd.NA rather than the string 'nan'.
    """
    for col in FORCE_STR_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str).replace('nan', pd.NA)
    return df


def cast_object_numerics(df):
    """
    Safety net: after concat, find any remaining object columns that are
    overwhelmingly numeric (>90% parseable) and convert them.
    Catches mixed int/str columns that slipped through earlier steps.
    """
    for col in df.select_dtypes(include='object').columns:
        # Skip columns we know should stay as strings
        if col in FORCE_STR_COLS:
            continue
        non_null = df[col].dropna()
        if len(non_null) == 0:
            continue
        converted = pd.to_numeric(
            non_null.astype(str).str.replace(r'[$,]', '', regex=True),
            errors='coerce'
        )
        if converted.notna().sum() / len(non_null) > 0.90:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(r'[$,]', '', regex=True),
                errors='coerce'
            )
    return df


# ── Normalization ─────────────────────────────────────────────────────────────

def normalize(df, quarter):
    """
    Apply all cross-quarter normalization to a freshly loaded dataframe.
    Safe to call on any sheet type (oon, qpa, air).
    """
    # 1. Strip whitespace from column names
    df.columns = [c.strip() for c in df.columns]

    # 2. Standardize column name variant
    #    "as Percent of QPA" (2023 Q1-Q2) -> "as % of QPA" (2023 Q3 onward)
    df.columns = [
        c.replace('as Percent of QPA', 'as % of QPA')
         .replace('As Percent of QPA', 'as % of QPA')
        for c in df.columns
    ]

    # 3. Replace CMS suppression/missing markers globally
    df = df.replace({'NR': pd.NA, 'N/R': pd.NA})

    # 4. Force code/description columns to string before any numeric conversion
    df = force_string_cols(df)

    # 5. Strip currency formatting and convert
    df = convert_currency_cols(df)

    # 6. Tag quarter
    df['Quarter'] = quarter

    return df


# ── Loaders ───────────────────────────────────────────────────────────────────

def load_xlsx_sheet(filepath, sheet_name, quarter):
    print(f"  Loading xlsx: {os.path.basename(filepath)} | sheet: {sheet_name}")
    df = pd.read_excel(filepath, sheet_name=sheet_name, engine='openpyxl')
    return normalize(df, quarter)


def load_csv(filepath, quarter):
    print(f"  Loading csv:  {os.path.basename(filepath)}")
    df = pd.read_csv(filepath, low_memory=False)
    return normalize(df, quarter)


def load_csv_chunks(filepaths, quarter):
    chunks = [load_csv(fp_path, quarter) for fp_path in filepaths]
    return pd.concat(chunks, ignore_index=True)


# ── Build functions ───────────────────────────────────────────────────────────

def build_oon():
    print("\n=== Building OON dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter, filetype, source = q['quarter'], q['filetype'], q['oon']
        print(f"\n{quarter}:")
        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_OON, quarter))
        else:
            all_dfs.append(load_csv_chunks([fp(f) for f in source], quarter))

    print("\nConcatenating all quarters...")
    oon = pd.concat(all_dfs, ignore_index=True)

    # Numeric conversions
    oon = convert_numeric_cols(oon, NUMERIC_PATTERNS)

    # Safety net for any remaining mixed-type object columns
    oon = cast_object_numerics(oon)

    # Domain mappings
    print("Applying domain mappings...")
    oon = apply_all_maps(oon)

    # Outcome bucket
    oon = add_outcome_bucket(oon)

    print(f"OON total rows:    {len(oon):,}")
    print(f"OON columns:       {len(oon.columns)}")
    print(f"OON quarters:      {oon['Quarter'].value_counts().sort_index().to_dict()}")
    print(f"OON outcome buckets:\n{oon['Outcome_Bucket'].value_counts().to_string()}")
    return oon


def build_qpa():
    print("\n=== Building QPA dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter, filetype, source = q['quarter'], q['filetype'], q['qpa']
        if source is None:
            print(f"{quarter}: no QPA file — skipping")
            continue
        print(f"\n{quarter}:")
        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_QPA, quarter))
        else:
            all_dfs.append(load_csv(fp(source), quarter))

    print("\nConcatenating all quarters...")
    qpa = pd.concat(all_dfs, ignore_index=True)

    # QPA sheet has dollar amount columns
    qpa = convert_numeric_cols(qpa, ['QPA', 'Offer', 'Prevailing', 'Percent'])
    qpa = cast_object_numerics(qpa)

    print(f"QPA total rows:    {len(qpa):,}")
    print(f"QPA quarters:      {qpa['Quarter'].value_counts().sort_index().to_dict()}")
    return qpa


def build_air():
    print("\n=== Building Air Ambulance dataset ===")
    all_dfs = []

    for q in QUARTERS:
        quarter, filetype, source = q['quarter'], q['filetype'], q['air']
        if source is None:
            print(f"{quarter}: no Air file — skipping")
            continue
        print(f"\n{quarter}:")
        if filetype == 'xlsx':
            all_dfs.append(load_xlsx_sheet(fp(source), SHEET_AIR, quarter))
        else:
            all_dfs.append(load_csv(fp(source), quarter))

    print("\nConcatenating all quarters...")
    air = pd.concat(all_dfs, ignore_index=True)

    # Numeric conversions
    air = convert_numeric_cols(air, NUMERIC_PATTERNS)
    air = cast_object_numerics(air)

    # Domain mappings
    print("Applying domain mappings...")
    air = apply_all_maps(air)

    # Outcome bucket
    air = add_outcome_bucket(air)

    print(f"Air total rows:    {len(air):,}")
    print(f"Air quarters:      {air['Quarter'].value_counts().sort_index().to_dict()}")
    return air


# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    oon = build_oon()
    qpa = build_qpa()
    air = build_air()

    # print("\n=== Saving parquet files ===")

    # oon_out = out('oon_all_quarters.parquet')
    # qpa_out = out('qpa_all_quarters.parquet')
    # air_out = out('air_all_quarters.parquet')

    # oon.to_parquet(oon_out, compression='snappy', index=False)
    # print(f"Saved: {oon_out}  ({os.path.getsize(oon_out)/1e6:.1f} MB)")

    # qpa.to_parquet(qpa_out, compression='snappy', index=False)
    # print(f"Saved: {qpa_out}  ({os.path.getsize(qpa_out)/1e6:.1f} MB)")

    # air.to_parquet(air_out, compression='snappy', index=False)
    # print(f"Saved: {air_out}  ({os.path.getsize(air_out)/1e6:.1f} MB)")

    # print("\nDone.")
    # print(f"  OON: {len(oon):,} rows  |  {len(oon.columns)} columns")
    # print(f"  QPA: {len(qpa):,} rows  |  {len(qpa.columns)} columns")
    # print(f"  Air: {len(air):,} rows  |  {len(air.columns)} columns")


=== Building OON dataset ===

2023Q1:
  Loading xlsx: 2023-q1-federal-idr-puf_0.xlsx | sheet: OON Emergency and Non-Emergency

2023Q2:
  Loading xlsx: Federal-IDR-PUF-for-2023-Q2.xlsx | sheet: OON Emergency and Non-Emergency

2023Q3:
  Loading xlsx: Federal IDR PUF for 2023 Q3.xlsx | sheet: OON Emergency and Non-Emergency

2023Q4:
  Loading xlsx: Federal IDR PUF for 2023 Q4.xlsx | sheet: OON Emergency and Non-Emergency

2024Q1:
  Loading xlsx: federal-idr-puf-for-2024-q1-as-of-march-18-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q2:
  Loading xlsx: federal-idr-puf-for-2024-q2-as-of-march-18-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q3:
  Loading xlsx: federal-idr-puf-for-2024-q3-as-of-may-28-2025.xlsx | sheet: OON Emergency and Non-Emergency

2024Q4:
  Loading xlsx: federal-idr-puf-for-2024-q4-as-of-may-28-2025.xlsx | sheet: OON Emergency and Non-Emergency

2025Q1:
  Loading csv:  Federal IDR PUF for 2025 Q1 - OON Emergency & Non-Emergency (As of January 21, 

In [133]:
qpa[qpa['QPA'].notna()].head()

,Service Code,Place of Service Code,Geographical Region,QPA,Provider/Facility Offer,Health Plan/Issuer Offer,Offer Selected from Provider or Issuer,Prevailing Offer,Default Decision,Quarter,Initiating Party,Dispute Line Item Type,Type of Service Code,Air Ambulance Pick-up Location
13,99285,23.0,"Abilene, TX",270.26,505.61,270.26,NaN,505.61,No,2023Q1,NaN,NaN,NaN,NaN
14,99285,23.0,"Abilene, TX",194.73,505.61,194.73,NaN,505.61,No,2023Q1,NaN,NaN,NaN,NaN
15,99284,23.0,"Abilene, TX",206.12,532.49,206.12,NaN,532.49,No,2023Q1,NaN,NaN,NaN,NaN
18,99283,23.0,"Abilene, TX",102.15,256.62,102.15,NaN,256.62,No,2023Q1,NaN,NaN,NaN,NaN
19,99283,23.0,"Abilene, TX",105.49,314.60,155.00,NaN,314.60,No,2023Q1,NaN,NaN,NaN,NaN


In [134]:
# oon[oon['Provider Email Domain'].str.contains('total', na=False)][[
#     'Provider Email Domain',
#     'Provider_Group'
# ]].drop_duplicates()

oon.groupby([
    'Outcome_Bucket',
]).agg(
    dispute_count=('Dispute Number', 'nunique')
).reset_index().sort_values(
    'dispute_count',
    ascending=False
)

,Outcome_Bucket,dispute_count
1,Contested Provider Win,1541801
3,Default Provider Win,406838
0,Contested Plan Win,253350
2,Default Plan Win,68052
4,Other / Split,1560


In [135]:
print("\n=== Saving parquet files ===")

oon_out = out('oon_all_quarters.parquet')
qpa_out = out('qpa_all_quarters.parquet')
air_out = out('air_all_quarters.parquet')

oon.to_parquet(oon_out, compression='snappy', index=False)
print(f"Saved: {oon_out}  ({os.path.getsize(oon_out)/1e6:.1f} MB)")

qpa.to_parquet(qpa_out, compression='snappy', index=False)
print(f"Saved: {qpa_out}  ({os.path.getsize(qpa_out)/1e6:.1f} MB)")

air.to_parquet(air_out, compression='snappy', index=False)
print(f"Saved: {air_out}  ({os.path.getsize(air_out)/1e6:.1f} MB)")

print("\nDone.")
print(f"  OON: {len(oon):,} rows  |  {len(oon.columns)} columns")
print(f"  QPA: {len(qpa):,} rows  |  {len(qpa.columns)} columns")
print(f"  Air: {len(air):,} rows  |  {len(air.columns)} columns")


=== Saving parquet files ===
Saved: ./parquet\oon_all_quarters.parquet  (195.4 MB)
Saved: ./parquet\qpa_all_quarters.parquet  (52.6 MB)


ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column Air Ambulance Vehicle Clinical Capacity Level with type object')

In [138]:
oon.head()

,Dispute Number,DLI Number,Payment Determination Outcome,Default Decision,Type of Dispute,Provider/Facility Group Name,Provider/Facility Name,Provider Email Domain,Provider/Facility NPI Number,Practice/Facility Size,...,Provider/Facility Offer as Percent of Median Provider/Facility Offer Amount,Health Plan/Issuer Offer as Percent of Median Health Plan/Issuer Offer Amount,Prevailing Offer as Percent of Median Prevailing Offer Amount,Quarter,Initiating Party,Provider_Group,Provider_Category,Carrier_Group,Anthem_Flag,Outcome_Bucket
0,DISP-135724,DLI - 582653,In Favor of Plan/Issuer,No,Single,LONGHORN EMERGENCY MEDICAL ASSOCIATES PA,"BLUDWORTH MD,WHITNEY G",teamhealth.com,1.649670e+09,101-500 Employees,...,0.99,1.00,1.00,2023Q1,NaN,TeamHealth,PE-Backed National,UnitedHealthcare,Other,Contested Plan Win
1,DISP-135637,DLI - 582530,In Favor of Plan/Issuer,No,Single,<NA>,ROXBOROUGH MEMORIAL HOSPITAL,primehealthcare.com,NaN,Over 500 Employees,...,1.00,1.00,1.00,2023Q1,NaN,Prime Healthcare,Hospital System,Aetna,Other,Contested Plan Win
2,DISP-135495,DLI - 582345,In Favor of Provider/Facility/AA Provider,No,Single,WOODLANDS EMERGENCY PHYSICIANS,WOODLANDS EMERGENCY PHYSICIANS,r1rcm.com,1.467990e+09,51-100 Employees,...,0.83,0.92,0.88,2023Q1,NaN,R1 RCM,Billing/Revenue Cycle,MultiPlan/Claritev,Other,Contested Provider Win
3,DISP-135654,DLI - 582547,In Favor of Provider/Facility/AA Provider,No,Single,APP OF TENNESSEE ED PLLC,APP OF TENNESSEE ED PLLC,r1rcm.com,1.003230e+09,101-500 Employees,...,0.71,1.15,0.71,2023Q1,NaN,R1 RCM,Billing/Revenue Cycle,Cigna,Other,Contested Provider Win
4,DISP-135828,DLI - 582852,In Favor of Provider/Facility/AA Provider,No,Batched,Remote Neuromonitoring Physicians PC,Remote Neuromonitoring Physicians PC,specialtycare.net,1.124253e+09,Over 500 Employees,...,1.00,NaN,1.00,2023Q1,NaN,SpecialtyCare,Specialty Group,UnitedHealthcare,Other,Contested Provider Win


In [143]:
oon.groupby([
    'Provider_Group',
]).agg(
    dispute_count=('Dispute Number', 'nunique')
).reset_index().sort_values(
    'dispute_count',
    ascending=False
)

Provider_Group    TeamHealthHaloMDSingleton AssociatesSCP Health...
dispute_count                                               2271462
dtype: object

In [153]:
oon[oon['Provider/Facility Name'].str.contains('wrfer', case=False, na=False)].head()

,Dispute Number,DLI Number,Payment Determination Outcome,Default Decision,Type of Dispute,Provider/Facility Group Name,Provider/Facility Name,Provider Email Domain,Provider/Facility NPI Number,Practice/Facility Size,...,Provider/Facility Offer as Percent of Median Provider/Facility Offer Amount,Health Plan/Issuer Offer as Percent of Median Health Plan/Issuer Offer Amount,Prevailing Offer as Percent of Median Prevailing Offer Amount,Quarter,Initiating Party,Provider_Group,Provider_Category,Carrier_Group,Anthem_Flag,Outcome_Bucket
23227,DISP-194868,DLI - 727121,In Favor of Plan/Issuer,No,Single,<NA>,"WRFER UNIVERSITY DRIVE, LLC",totalcare.us,1.124603e+09,20-50 Employees,...,3.54,2.52,1.7,2023Q1,NaN,TotalCare / Radiology Partners,PE-Backed National,BCBS Texas,Other,Contested Plan Win
23228,DISP-194868,DLI - 727147,In Favor of Plan/Issuer,No,Single,<NA>,"WRFER UNIVERSITY DRIVE, LLC",totalcare.us,1.124603e+09,20-50 Employees,...,NaN,NaN,NaN,2023Q1,NaN,TotalCare / Radiology Partners,PE-Backed National,BCBS Texas,Other,Contested Plan Win
23229,DISP-194868,DLI - 727157,In Favor of Plan/Issuer,No,Single,<NA>,"WRFER UNIVERSITY DRIVE, LLC",totalcare.us,1.124603e+09,20-50 Employees,...,NaN,NaN,NaN,2023Q1,NaN,TotalCare / Radiology Partners,PE-Backed National,BCBS Texas,Other,Contested Plan Win
23230,DISP-194868,DLI - 727168,In Favor of Plan/Issuer,No,Single,<NA>,"WRFER UNIVERSITY DRIVE, LLC",totalcare.us,1.124603e+09,20-50 Employees,...,NaN,NaN,NaN,2023Q1,NaN,TotalCare / Radiology Partners,PE-Backed National,BCBS Texas,Other,Contested Plan Win
23231,DISP-194868,DLI - 727197,In Favor of Plan/Issuer,No,Single,<NA>,"WRFER UNIVERSITY DRIVE, LLC",totalcare.us,1.124603e+09,20-50 Employees,...,NaN,NaN,NaN,2023Q1,NaN,TotalCare / Radiology Partners,PE-Backed National,BCBS Texas,Other,Contested Plan Win
